# DB7-032 — JOPS-inspired learning for the existing Hudgins BRB

## Research question
Can joint structure and parameter optimization improve our existing hierarchical
BRB on NinaPro DB7 **EMG**? The user confirmed this dataset; this is not an EEG study.

This is a **budgeted hierarchical JOPS-inspired classification adaptation**, not
an exact replication of Yang et al.'s regression experiments. It preserves the
12-channel hierarchical BRB and ER fusion, and jointly searches reference counts,
reference locations, rule weights, attribute weights, consequent beliefs and
fusion weights. It does not add neural branches or train an SI-to-I correction gate.

## Fixed data and inputs
- Subjects S1–S20; E1 restimulus labels 1–17; rest excluded.
- Training repetitions 1, 3, 4 and 6; reserved test repetitions 2 and 5.
- Each EMG window is 200 ms (400 samples at 2 kHz).
- Training stride is 50 ms; test stride is 10 ms, matching DB7-031.
- Per labeled repetition, apply a fourth-order 20–450 Hz Butterworth filter and
  a 50 Hz, Q=30 notch, both zero-phase. Windows do not cross repetitions.
- Each of the 12 EMG channels contributes MAV, WL, ZC and SSC: all 48 inputs remain.
- The unchanged DB7-030 preprocessing and DB7-031 data helpers are embedded below.
  Raw recording hashes and final training-derived threshold choices are checked
  against the saved parent evidence. Test data do not fit preprocessing.

## Model structure
Each of 12 channel modules receives its four Hudgins attributes. The AND-style
matching and complete-belief ER within channels and between channels are retained.
A channel uses the Cartesian product of its feature references as its rule base.
Each rule predicts 17 gesture beliefs; the final output is one 17-class distribution.

The allowed reference count for each of the 48 attributes is 2, 3 or 4. Thus the
number of rules in a channel is the product of its four reference counts. A
three-reference channel has 81 rules; two references each give 16 rules. These
are compact channel modules, not a single infeasible 48-attribute Cartesian grid.

## Five candidate structures per subject and seed
The fixed search budget is:
1. The existing all-three-reference structure.
2. An all-two-reference structure.
3. Three further SOHS-inspired add/prune proposals, keeping every attribute within
   the range 2–4 references.

A Pareto archive removes a candidate when another has no worse validation
accuracy and no more references in **every one of the 48 attributes**, with at
least one strict improvement. This componentwise reference-count comparison is
different from dominance based only on total rule count. Candidate comparisons
use only the four training repetitions. Structure
selection maximizes mean held-repetition accuracy; ties use lower classification
negative log likelihood, then fewer rules. There is no test-based rule-count choice.

This small structure search is a budget limit, not proof that its chosen structure
is globally optimal. It tests whether a practical JOPS adaptation improves this
specific BRB; it does not prove the full search space has been exhausted.

## Parameters learned by Differential Evolution
For every fitting split:
- The low and high feature endpoints are the fitting-data 5th and 95th percentiles.
  Internal reference positions are optimized while maintaining strict order.
  A two-reference attribute has no internal reference to optimize.
- Rule consequent beliefs are initialized from fitting-data rule/class evidence.
- Rule weights, attribute weights, consequent beliefs and channel-fusion weights
  are trained, with constraints enforced by the parameterization.
- The fitting objective is classification cross-entropy. Gesture numbers are
  categorical labels; numerical distance between gesture IDs is not a loss.

Optimization uses cooperative/block Differential Evolution: **population 12,
four generations per block, two cycles over the 12 channel blocks plus the fusion
block**. Fitness uses the full fitting dataset. These budgets are prespecified,
not selected by checking test accuracy. Learning histories record accepted
updates and stagnation, including a block where no proposal improved the loss.
The mutation factor is 0.5 and crossover probability is 0.9. GPU fitness is
evaluated in chunks of 2,048 windows and four population members; chunking does
not subsample or change the full-dataset objective.

This differs from the paper's full-vector DE and regression complexity criterion.
The budgeted block search makes the hierarchical classification experiment
practical and keeps its actual changes inspectable.

**Initialization also changes from DB7-031:** that model initialized every rule
with the fitting-class prior; this experiment initializes each rule using its
own fitting-data activation/class evidence. Therefore the historical comparison
changes initialization, reference positions/counts and optimization together.
An improvement cannot be attributed to DE or structure search alone. The exported
initial/final rule tables and optimization histories show how much the subsequent
search changes its empirical initializer.

## Repetition validation and final fitting
For each candidate, hold out one of training repetitions 1/3/4/6 and fit on the
other three. Repeat four times. Feature scales, references, rule initialization
and nested ZC/SSC threshold screening exclude the held-out repetition.

After selecting a structure using mean validation results, fit it from scratch
on all four training repetitions. Only then predict repetitions 2/5. The test
labels are used to calculate metrics, never to select a structure or parameters.

## Compute and reproducibility
There are 20 subjects × three seeds (42, 43, 44) × five candidates × four folds =
**1,200 validation BRB fits**, followed by **60 final fits**. The 300 candidate
evaluations include the prespecified initial structures and the add/prune proposals.

Two worker processes divide subjects between cuda:0 and cuda:1. The notebook
asserts that **both Kaggle T4 GPUs** are available, and records each worker's GPU
name and assigned subjects. An independent CPU self-test checks memberships, ER
and constrained optimization before the full experiment.

The first full fitting run on each worker provides a rough runtime forecast.
If its extrapolated worker compute exceeds **nine hours**, the run stops and
preserves partial diagnostics; it does not silently reduce the search budget.
That outcome calls for splitting the same protocol into smaller runs. The
GitHub workflow monitors for five hours within its six-hour job; a longer Kaggle
run can be monitored again with `monitor_ref` without submitting duplicate work.
The forecast is approximate because later candidate rule counts can differ.

## What the results can establish
The matched comparator is the same filtered 48-feature LDA. DB7-031 supplies the
previous hierarchical BRB result for comparison. SI also uses inertial inputs,
so SI accuracy is contextual rather than a modality-matched baseline.

Report mean subject accuracy, pooled window accuracy, macro-F1, per-gesture errors,
confusions and paired subject-level uncertainty. Overlapping windows and the
three seeds are not independent experimental subjects. These previously inspected
recordings provide exploratory evidence; an apparent improvement is not an
independent confirmation of generalization.

Exports include candidate structures, fitting/validation histories, selection
evidence, initial and trained rules/references/weights, final predictions,
probabilities and completion checks. No accuracy is promised before execution.


## 1. Preserve feature extraction

Unchanged filtering, windowing and Hudgins feature definitions.


In [ ]:
%%writefile /kaggle/working/db7_030_hudgins.py
"""DB7-030: Hudgins TD4 complementarity against frozen DB7-016 SI/I outputs.

The neural predictions are inputs, never re-fitted. Every TD4 fitting or
selection decision uses repetitions 1/3/4/6; repetitions 2/5 are evaluated
only after those choices are fixed. Overlapping windows stay in their trial.
"""
from __future__ import annotations

import argparse
import hashlib
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, iirnotch, sosfiltfilt
from scipy.stats import rankdata

FS = 2000
WINDOW = 400
TRAIN_STEP = 100  # 50 ms: reduce duplicated training evidence
TEST_STEP = 20  # 10 ms: match the frozen SI/I prediction grid
TRAIN_REPS = (1, 3, 4, 6)
TEST_REPS = (2, 5)
FAMILIES = ("MAV", "WL", "ZC", "SSC")
THRESHOLD_RATIOS = (0.005, 0.01, 0.02)
KEY = ["subject", "gesture", "native_repetition", "window_start", "window_end"]


def find_root() -> Path:
    for parent in (Path("/kaggle/input"), Path("D:/NinaProDB7")):
        if parent.exists():
            found = sorted(parent.rglob("S1_E1_A1.mat"))
            if found:
                return found[0].parent.parent
    raise FileNotFoundError("NinaPro DB7 E1 recordings are not attached")


def load_subject(root: Path, subject: int):
    candidates = list(root.rglob(f"S{subject}_E1_A1.mat"))
    if len(candidates) != 1:
        raise ValueError(f"S{subject}: expected one E1 file, found {len(candidates)}")
    data = loadmat(candidates[0], variable_names=["emg", "restimulus", "rerepetition"])
    emg = np.asarray(data["emg"], dtype=np.float32)
    if emg.shape[1] != 12 and emg.shape[0] == 12:
        emg = emg.T
    labels = np.asarray(data["restimulus"]).ravel().astype(int)
    repetitions = np.asarray(data["rerepetition"]).ravel().astype(int)
    if emg.shape[1] != 12 or len(emg) != len(labels) or len(emg) != len(repetitions):
        raise ValueError(f"S{subject}: invalid EMG/annotation alignment")
    if set(np.unique(labels)) != set(range(18)):
        raise ValueError(f"S{subject}: E1 labels 0..17 are not present")
    sos = butter(4, [20, 450], btype="bandpass", fs=FS, output="sos")
    b, a = iirnotch(50, 30, fs=FS)
    edges = np.r_[0, np.flatnonzero(np.diff(labels)) + 1, len(labels)]
    runs = []
    for start, end in zip(edges[:-1], edges[1:]):
        gesture = int(labels[start])
        if gesture == 0:
            continue
        unique_rep = np.unique(repetitions[start:end])
        if len(unique_rep) != 1 or unique_rep[0] not in (*TRAIN_REPS, *TEST_REPS):
            raise ValueError(f"S{subject} gesture {gesture}: invalid repetition")
        filtered = sosfiltfilt(sos, emg[start:end], axis=0)
        filtered = filtfilt(b, a, filtered, axis=0).astype(np.float32)
        runs.append((gesture, int(unique_rep[0]), int(start), filtered))
    counts = pd.Series([(g, r) for g, r, _, _ in runs]).value_counts()
    if len(counts) != 102 or not (counts == 1).all():
        raise ValueError(f"S{subject}: expected 17 gestures x 6 repetitions")
    digest = hashlib.sha256(emg.tobytes()).hexdigest()
    return runs, {"file": str(candidates[0]), "raw_emg_sha256": digest}


def threshold_scale(runs, fitting_repetitions=TRAIN_REPS):
    """Estimate channel amplitudes using only the repetitions used for fitting."""
    train_signal = np.concatenate([np.abs(x) for _, r, _, x in runs
                                   if r in fitting_repetitions])
    return np.maximum(np.percentile(train_signal, 95, axis=0), 1e-12)


def extract(runs, subject, split, threshold):
    features, rows = [], []
    step = TRAIN_STEP if split == "train" else TEST_STEP
    reps = TRAIN_REPS if split == "train" else TEST_REPS
    for gesture, rep, absolute_start, signal in runs:
        if rep not in reps or len(signal) < WINDOW:
            continue
        # Shape: windows x channels x 400 samples. Each run is handled alone.
        windows = np.lib.stride_tricks.sliding_window_view(signal, WINDOW, axis=0)[::step]
        mav = np.abs(windows).mean(axis=-1)
        difference = np.diff(windows, axis=-1)
        wl = np.abs(difference).sum(axis=-1)
        zc = ((windows[..., :-1] * windows[..., 1:] < 0) &
              (np.abs(difference) >= threshold[None, :, None])).sum(axis=-1)
        ssc = ((difference[..., :-1] * difference[..., 1:] < 0) &
               (np.maximum(np.abs(difference[..., :-1]),
                           np.abs(difference[..., 1:])) >= threshold[None, :, None])).sum(axis=-1)
        features.append(np.concatenate([mav, wl, zc, ssc], axis=1).astype(np.float32))
        offsets = np.arange(len(windows)) * step
        rows.extend((subject, gesture, rep, absolute_start + int(k), absolute_start + int(k) + WINDOW)
                    for k in offsets)
    frame = pd.DataFrame(rows, columns=KEY)
    if frame.duplicated(KEY).any():
        raise AssertionError("Duplicate feature windows")
    return np.concatenate(features), frame


def columns(mask):
    return np.concatenate([np.arange(12 * i, 12 * (i + 1)) for i in range(4) if mask & (1 << i)])


def fit_lda(x, y):
    """Train-only robust scaling and regularized pooled-covariance LDA."""
    x = np.asarray(x, dtype=np.float64)
    center = np.median(x, axis=0)
    scale = np.percentile(x, 75, axis=0) - np.percentile(x, 25, axis=0)
    scale = np.maximum(scale, np.maximum(np.max(np.abs(x), axis=0) * 1e-6, 1e-9))
    z = (x - center) / scale
    classes = np.arange(1, 18)
    counts = np.bincount(y, minlength=18)[1:]
    if np.any(counts == 0):
        raise ValueError("LDA training is missing a gesture")
    means = np.stack([z[y == g].mean(axis=0) for g in classes])
    residual = z - means[y - 1]
    covariance = residual.T @ residual / max(len(z) - len(classes), 1)
    diagonal = np.diag(np.diag(covariance))
    covariance = 0.9 * covariance + 0.1 * diagonal
    covariance.flat[::len(covariance) + 1] += 1e-5
    weights = np.linalg.solve(covariance, means.T)
    bias = -0.5 * np.sum(means * weights.T, axis=1) + np.log(counts / counts.sum())
    return center, scale, weights, bias


def predict(model, x):
    center, scale, weights, bias = model
    score = ((np.asarray(x, dtype=np.float64) - center) / scale) @ weights + bias
    score -= score.max(axis=1, keepdims=True)
    prob = np.exp(score)
    prob /= prob.sum(axis=1, keepdims=True)
    return prob.argmax(axis=1).astype(np.int16) + 1, prob


def cv_score(fold_features, frame, mask):
    """Each fold has features extracted with its own fitting-only threshold."""
    y = frame.gesture.to_numpy(int)
    rep = frame.native_repetition.to_numpy(int)
    ix = columns(mask)
    scores = []
    for held in TRAIN_REPS:
        train, val = rep != held, rep == held
        x = fold_features[held]
        pred, _ = predict(fit_lda(x[train][:, ix], y[train]), x[val][:, ix])
        scores.append(float(np.mean(pred == y[val])))
    return float(np.mean(scores))


def check_trace_coverage(subject_trace, test_meta, subject):
    """Require exactly one prediction per raw test window for each saved seed."""
    if set(subject_trace.seed.unique()) != {42, 43, 44}:
        raise AssertionError(f"S{subject}: expected exactly seeds 42, 43 and 44")
    expected = pd.MultiIndex.from_frame(test_meta[KEY])
    for seed, group in subject_trace.groupby("seed"):
        actual = pd.MultiIndex.from_frame(group[KEY])
        if actual.has_duplicates or len(actual) != len(expected):
            raise AssertionError(f"S{subject} seed {seed}: duplicate/missing test windows")
        if len(expected.difference(actual)) or len(actual.difference(expected)):
            raise AssertionError(f"S{subject} seed {seed}: raw and saved test keys differ")


def subset_complementarity(joined, prediction, margin):
    """Count useful TD4 evidence and harmful switches for one subject and seed."""
    truth = joined.gesture.to_numpy(int)
    si = joined.si_pred.to_numpy(int)
    inertial = joined.i_pred.to_numpy(int)
    si_correct, i_correct = si == truth, inertial == truth
    good = prediction == truth
    shared = ~si_correct & ~i_correct
    recovery, harm = ~si_correct & i_correct, si_correct & ~i_correct
    switch = (si != inertial) & (margin > 0)
    recovered, harmed = int((switch & recovery).sum()), int((switch & harm).sum())
    return dict(windows=len(truth), td_correct=int(good.sum()),
        td_accuracy=float(good.mean()), si_correct=int(si_correct.sum()),
        si_accuracy=float(si_correct.mean()), shared_wrong=int(shared.sum()),
        shared_recovered=int((shared & good).sum()),
        si_wrong_td_correct=int((~si_correct & good).sum()),
        si_correct_td_wrong=int((si_correct & ~good).sum()),
        recovery_opportunities=int(recovery.sum()), harm_opportunities=int(harm.sum()),
        td_correct_on_recovery=int((recovery & good).sum()),
        td_correct_on_harm=int((harm & good).sum()),
        switches=int(switch.sum()), switch_recovered=recovered, switch_harmed=harmed,
        switch_net_corrected=recovered - harmed,
        switch_accuracy=float((si_correct.sum() + recovered - harmed) / len(truth)),
        si_i_td_oracle_correct=int((si_correct | i_correct | good).sum()))


def subject_study(root, subject, trace, out):
    runs, source = load_subject(root, subject)
    scale = threshold_scale(runs)
    # A validation repetition must not affect the ZC/SSC amplitude thresholds.
    fold_scales = {held: threshold_scale(runs, tuple(r for r in TRAIN_REPS if r != held))
                   for held in TRAIN_REPS}
    threshold_cv = []
    training = {}
    cv_features = {}
    for ratio in THRESHOLD_RATIOS:
        x, meta = extract(runs, subject, "train", scale * ratio)
        training[ratio] = x
        cv_features[ratio] = {}
        for held in TRAIN_REPS:
            fold_x, fold_meta = extract(runs, subject, "train", fold_scales[held] * ratio)
            if not fold_meta.equals(meta):
                raise AssertionError("Fold feature windows changed with the noise threshold")
            cv_features[ratio][held] = fold_x
        score = cv_score(cv_features[ratio], meta, 15)
        threshold_cv.append(dict(subject=subject, ratio=ratio, td4_cv_accuracy=score))
    selected_ratio = sorted(THRESHOLD_RATIOS,
                            key=lambda v: (-next(r["td4_cv_accuracy"] for r in threshold_cv if r["ratio"] == v),
                                           abs(v - 0.01)))[0]
    x_train = training[selected_ratio]
    selected_cv_features = cv_features[selected_ratio]
    del training
    del cv_features
    x_test, test_meta = extract(runs, subject, "test", scale * selected_ratio)
    y_train = meta.gesture.to_numpy(int)
    y_test = test_meta.gesture.to_numpy(int)
    subset_cv = []
    for mask in range(1, 16):
        subset_cv.append(dict(subject=subject, mask=mask,
                              families="+".join(FAMILIES[i] for i in range(4) if mask & (1 << i)),
                              cv_accuracy=cv_score(selected_cv_features, meta, mask)))
    del selected_cv_features
    chosen = sorted(subset_cv, key=lambda r: (-r["cv_accuracy"], int(r["mask"]).bit_count(), r["mask"]))[0]["mask"]
    test_rows, predictions = [], {}
    for row in subset_cv:
        mask = row["mask"]
        ix = columns(mask)
        pred, prob = predict(fit_lda(x_train[:, ix], y_train), x_test[:, ix])
        predictions[mask] = (pred, prob)
        test_rows.append({**row, "test_accuracy": float(np.mean(pred == y_test)),
                          "correct": int(np.sum(pred == y_test)), "windows": len(y_test),
                          "cv_selected": mask == chosen, "td4_full": mask == 15,
                          "selected_threshold_ratio": selected_ratio})
    subject_trace = trace[trace.subject == subject].copy()
    check_trace_coverage(subject_trace, test_meta, subject)
    joined = subject_trace.merge(test_meta.assign(feature_row=np.arange(len(test_meta))),
                                 on=KEY, how="left", validate="many_to_one")
    if joined.feature_row.isna().any() or len(joined) != len(subject_trace):
        raise AssertionError(f"S{subject}: saved neural/test feature window mismatch")
    if not np.array_equal(joined.gesture.to_numpy(), y_test[joined.feature_row.to_numpy(int)]):
        raise AssertionError("Label mismatch after joining predictions")
    combination_rows = []
    indices = joined.feature_row.to_numpy(int)
    si, inertial = joined.si_pred.to_numpy(int), joined.i_pred.to_numpy(int)
    for row in subset_cv:
        mask = row["mask"]
        pred, prob = predictions[mask]
        aligned_pred = pred[indices]
        margin = prob[indices, inertial - 1] - prob[indices, si - 1]
        for seed, positions in joined.groupby("seed").indices.items():
            positions = np.asarray(positions)
            combination_rows.append(dict(subject=subject, seed=int(seed), mask=mask,
                families=row["families"], cv_selected=mask == chosen,
                **subset_complementarity(joined.iloc[positions], aligned_pred[positions],
                                         margin[positions])))
    for name, mask in (("selected", chosen), ("TD4", 15)):
        pred, prob = predictions[mask]
        indices = joined.feature_row.to_numpy(int)
        si = joined.si_pred.to_numpy(int)
        inertial = joined.i_pred.to_numpy(int)
        joined[f"{name}_pred"] = pred[indices]
        joined[f"{name}_correct"] = pred[indices] == joined.gesture.to_numpy(int)
        # Candidate support is diagnostic; do not treat raw LDA probabilities as calibrated.
        joined[f"{name}_candidate_margin"] = (prob[indices, inertial - 1] -
                                               prob[indices, si - 1]).astype(np.float32)
    joined.drop(columns="feature_row").to_csv(out / f"S{subject:02}_window_predictions.csv.gz", index=False)
    (out / f"S{subject:02}_source.json").write_text(json.dumps({**source, "subject": subject,
        "threshold_ratio": selected_ratio, "selected_subset": chosen,
        "threshold_scale_final": scale.tolist(),
        "threshold_scale_by_cv_held_repetition": {str(r): v.tolist() for r, v in fold_scales.items()},
        "train_windows_50ms": len(x_train), "test_windows_10ms": len(x_test)}, indent=2))
    print(f"S{subject:02}: train {len(x_train)}, test {len(x_test)}, "
          f"TD4 {test_rows[14]['test_accuracy']:.4f}, chosen {chosen:04b}", flush=True)
    return threshold_cv, subset_cv, test_rows, combination_rows


def summary(out, subjects):
    parts = [pd.read_csv(out / f"S{s:02}_window_predictions.csv.gz") for s in subjects]
    data = pd.concat(parts, ignore_index=True)
    if data.duplicated(KEY + ["seed"]).any():
        raise AssertionError("Duplicate test predictions")
    expected = len(data)
    target = data.gesture.to_numpy()
    correct = {"SI": data.si_pred.to_numpy() == target, "I": data.i_pred.to_numpy() == target,
               "TD4": data.TD4_pred.to_numpy() == target,
               "selected": data.selected_pred.to_numpy() == target}
    si_wrong = ~correct["SI"]
    shared = si_wrong & ~correct["I"]
    recovery = si_wrong & correct["I"]
    prevent_harm = correct["SI"] & ~correct["I"]
    records = []
    gate_rows = []
    for method in ("TD4", "selected"):
        good = correct[method]
        records.append(dict(method=method, windows=expected, correct=int(good.sum()),
            accuracy=float(good.mean()), si_errors=int(si_wrong.sum()),
            si_i_both_wrong=int(shared.sum()), shared_errors_recovered=int((shared & good).sum()),
            shared_errors_recovery_pct=float(100 * (shared & good).sum() / shared.sum()),
            si_wrong_i_correct_support=int((recovery & good).sum()),
            si_correct_i_wrong_td_correct=int((prevent_harm & good).sum()),
            si_correct_td_wrong=int((correct["SI"] & ~good).sum()),
            si_i_td_oracle_accuracy=float((correct["SI"] | correct["I"] | good).mean()),
            si_td_oracle_accuracy=float((correct["SI"] | good).mean())))
        margin = data[f"{method}_candidate_margin"].to_numpy(float)
        switch = (data.si_pred.to_numpy() != data.i_pred.to_numpy()) & (margin > 0)
        recovered = int((switch & recovery).sum())
        harmed = int((switch & prevent_harm).sum())
        directional = recovery | prevent_harm
        ranks = rankdata(margin[directional], method="average")
        positive = recovery[directional]
        negative_count = int((~positive).sum())
        auc = ((ranks[positive].sum() - positive.sum() * (positive.sum() + 1) / 2)
               / (positive.sum() * negative_count))
        gate_rows.append(dict(method=method, disagreement_windows=int((data.si_pred != data.i_pred).sum()),
            recovery_opportunities=int(recovery.sum()), harm_opportunities=int(prevent_harm.sum()),
            pair_margin_auc_recovery_vs_harm=float(auc),
            fixed_zero_margin_switches=int(switch.sum()), recovered=recovered, harmed=harmed,
            net_corrected=recovered - harmed,
            fixed_zero_margin_gate_accuracy=float((correct["SI"].sum() + recovered - harmed) / expected),
            note="Fixed threshold zero; diagnostic only, no BRB fitting or threshold tuning"))
    pd.DataFrame(records).to_csv(out / "complementarity_summary.csv", index=False)
    pd.DataFrame(gate_rows).to_csv(out / "candidate_preference_diagnostic.csv", index=False)
    for grouping, filename in [(["subject", "seed"], "subject_seed_complementarity.csv"),
                               (["subject", "gesture"], "subject_gesture_complementarity.csv"),
                               (["gesture"], "gesture_complementarity.csv"),
                               (["phase"], "phase_complementarity.csv")]:
        rows = []
        for key, indices in data.groupby(grouping, sort=True).indices.items():
            ix = np.asarray(indices)
            values = key if isinstance(key, tuple) else (key,)
            row = dict(zip(grouping, values))
            row.update(windows=len(ix), SI_accuracy=float(correct["SI"][ix].mean()),
                       I_accuracy=float(correct["I"][ix].mean()))
            for method in ("TD4", "selected"):
                row[f"{method}_accuracy"] = float(correct[method][ix].mean())
                row[f"{method}_shared_recovered"] = int((shared & correct[method])[ix].sum())
                row[f"{method}_si_error_recovered"] = int((si_wrong & correct[method])[ix].sum())
                row[f"{method}_si_correct_harmed"] = int((correct["SI"] & ~correct[method])[ix].sum())
            row["SI_wrong"] = int(si_wrong[ix].sum())
            row["shared_wrong"] = int(shared[ix].sum())
            rows.append(row)
        pd.DataFrame(rows).to_csv(out / filename, index=False)
    selected = pd.read_csv(out / "subset_test_metrics.csv")
    aggregate = selected.groupby(["mask", "families"]).agg(mean_subject_test_accuracy=("test_accuracy", "mean"),
        mean_subject_cv_accuracy=("cv_accuracy", "mean"), subjects_selected=("cv_selected", "sum")).reset_index()
    aggregate.sort_values("mean_subject_cv_accuracy", ascending=False).to_csv(out / "feature_combination_summary.csv", index=False)
    combinations = pd.read_csv(out / "subset_subject_seed_complementarity.csv")
    count_fields = ["windows", "td_correct", "si_correct", "shared_wrong", "shared_recovered",
        "si_wrong_td_correct", "si_correct_td_wrong", "recovery_opportunities", "harm_opportunities",
        "td_correct_on_recovery", "td_correct_on_harm", "switches", "switch_recovered", "switch_harmed",
        "switch_net_corrected", "si_i_td_oracle_correct"]
    rows = []
    for (mask, families), group in combinations.groupby(["mask", "families"]):
        row = dict(mask=int(mask), families=families,
                   **{field: int(group[field].sum()) for field in count_fields})
        row.update(mean_subject_td_accuracy=float(group.td_accuracy.mean()),
            mean_subject_si_accuracy=float(group.si_accuracy.mean()),
            mean_subject_switch_accuracy=float(group.switch_accuracy.mean()),
            pooled_td_accuracy=row["td_correct"] / row["windows"],
            pooled_switch_accuracy=(row["si_correct"] + row["switch_net_corrected"]) / row["windows"],
            shared_recovery_pct=100 * row["shared_recovered"] / max(row["shared_wrong"], 1),
            pooled_si_i_td_oracle_accuracy=row["si_i_td_oracle_correct"] / row["windows"])
        rows.append(row)
    pd.DataFrame(rows).to_csv(out / "subset_complementarity_summary.csv", index=False)
    completion = dict(success=True, subjects=subjects, seeds=[42, 43, 44],
        test_window_seed_evaluations=expected, neural_fits=0,
        lda_final_fits=15 * len(subjects), lda_cv_fits=(3 + 15) * 4 * len(subjects),
        exact_seed_window_coverage_verified=True, cv_threshold_scale_fitting_repetitions_only=True,
        protocol="DB7-016 200ms/10ms test; 50ms TD4 training grid; repetition-grouped train-only fourfold selection")
    (out / "completion.json").write_text(json.dumps(completion, indent=2))
    return completion


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, default=None)
    parser.add_argument("--trace", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--subjects", type=int, nargs="+", default=list(range(1, 21)))
    args = parser.parse_args()
    root = args.input or find_root()
    args.out.mkdir(parents=True, exist_ok=True)
    trace = pd.read_csv(args.trace)
    needed = KEY + ["seed", "si_pred", "i_pred", "phase"]
    if any(v not in trace for v in needed):
        raise ValueError("Saved SI/I trace is missing required columns")
    if trace.duplicated(KEY + ["seed"]).any():
        raise ValueError("Saved SI/I trace has duplicate windows")
    source = dict(raw_root=str(root), trace_file=str(args.trace),
                  trace_sha256=hashlib.sha256(args.trace.read_bytes()).hexdigest(),
                  subjects=args.subjects, train_repetitions=TRAIN_REPS,
                  test_repetitions=TEST_REPS, window_samples=WINDOW,
                  train_step_samples=TRAIN_STEP, test_step_samples=TEST_STEP,
                  feature_families=FAMILIES, threshold_ratios=THRESHOLD_RATIOS,
                  lda_covariance_shrinkage=0.1,
                  cv_threshold_scale="95th percentile fitted separately on three fitting repetitions",
                  final_threshold_scale="95th percentile fitted on repetitions 1/3/4/6",
                  testing_note="Existing inspected DB7-016 test split; exploratory, no independent confirmation")
    (args.out / "PROTOCOL.json").write_text(json.dumps(source, indent=2))
    threshold_rows, cv_rows, test_rows, combination_rows = [], [], [], []
    for s in args.subjects:
        a, b, c, d = subject_study(root, s, trace, args.out)
        threshold_rows.extend(a)
        cv_rows.extend(b)
        test_rows.extend(c)
        combination_rows.extend(d)
        pd.DataFrame(threshold_rows).to_csv(args.out / "threshold_cv.csv", index=False)
        pd.DataFrame(cv_rows).to_csv(args.out / "subset_cv.csv", index=False)
        pd.DataFrame(test_rows).to_csv(args.out / "subset_test_metrics.csv", index=False)
        pd.DataFrame(combination_rows).to_csv(args.out / "subset_subject_seed_complementarity.csv", index=False)
    print(json.dumps(summary(args.out, args.subjects), indent=2), flush=True)


if __name__ == "__main__":
    main()


## 2. Preserve parent data and validation helpers

The previous BRB code is embedded for shared helpers; the new orchestrator runs the JOPS engine.


In [ ]:
%%writefile /kaggle/working/db7_031_brb.py
"""DB7-031: train a hierarchical 48-input, three-reference BRB classifier.

Twelve channel BRBs each contain 3^4=81 rules. Their 17-class beliefs are
combined using ER. Consequents, rule/attribute weights and fusion weights
are jointly optimized. Test repetitions never select parameters or epochs.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import argparse, itertools, json, time, subprocess, sys, zipfile, hashlib
import numpy as np
import pandas as pd
import db7_030_hudgins as features

SEEDS=(42,43,44)
MAX_EPOCHS=60
BATCH=512
LR=.02
PENALTY=.001
NAMES=[f'{f}_CH{c:02}' for f in features.FAMILIES for c in range(1,13)]
BITS=np.array(list(itertools.product(range(3),repeat=4)),dtype=np.int64)
ACTIVE_BITS=np.array(list(itertools.product(range(2),repeat=4)),dtype=np.int64)

def write(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')

def reference_values(x):
    """Three raw-feature references, fitted using fitting repetitions only."""
    r=np.percentile(x,[5,50,95],axis=0).T
    span=np.maximum(r[:,2]-r[:,0],np.maximum(np.max(np.abs(x),axis=0)*1e-6,1e-12))
    tiny=span*1e-6
    r[:,0]=np.minimum(r[:,0],r[:,1]-tiny)
    r[:,2]=np.maximum(r[:,2],r[:,1]+tiny)
    assert r.shape==(48,3) and np.all(np.diff(r,axis=1)>0)
    return r

def encode(x,refs):
    """Only 16 of 81 rules can activate per channel; retain matching degrees."""
    x=np.asarray(x,float).reshape(-1,4,12).transpose(0,2,1)
    r=refs.reshape(4,12,3).transpose(1,0,2)
    low=(x>=r[None,:,:,1]).astype(np.int64)
    a=np.where(low==0,r[None,:,:,0],r[None,:,:,1])
    b=np.where(low==0,r[None,:,:,1],r[None,:,:,2])
    frac=np.clip((x-a)/(b-a),0,1)
    matches=np.where(ACTIVE_BITS[None,None,:,:]==0,1-frac[:,:,None,:],frac[:,:,None,:])
    index=((low[:,:,None,:]+ACTIVE_BITS[None,None,:,:])*np.array([27,9,3,1])).sum(-1)
    valid=np.all(matches>0,axis=-1)
    assert valid.any(axis=-1).all()
    return index,np.log(np.maximum(matches,1e-30)).astype(np.float32),valid

def er_numpy(a,beta,axis):
    product=np.prod(1+a[...,None]*(beta-1),axis=axis)
    ignorance=np.prod(1-a,axis=axis)
    v=product-ignorance[...,None]
    return v/v.sum(-1,keepdims=True)

def make_model(prior):
    import torch
    from torch import nn
    class HierarchicalBRB(nn.Module):
        def __init__(self):
            super().__init__()
            initial=torch.tensor(np.log(prior),dtype=torch.float32).repeat(12,81,1)
            self.belief_logits=nn.Parameter(initial.clone())
            self.register_buffer('initial_logits',initial)
            self.rule_logits=nn.Parameter(torch.zeros(12,81))
            self.attribute_logits=nn.Parameter(torch.zeros(12,4))
            self.fusion_logits=nn.Parameter(torch.zeros(12))
        def weights(self):
            aw=self.attribute_logits.exp()
            aw=aw/aw.amax(dim=-1,keepdim=True)
            return self.belief_logits.softmax(-1),self.rule_logits.exp(),aw,self.fusion_logits.softmax(-1)
        @staticmethod
        def er(a,beta,dim):
            v=torch.prod(1+a.unsqueeze(-1)*(beta-1),dim=dim)-torch.prod(1-a,dim=dim).unsqueeze(-1)
            v=v.clamp_min(1e-12)
            return v/v.sum(-1,keepdim=True)
        def forward(self,index,logmatches,valid):
            beta,rw,aw,fw=self.weights()
            channel=torch.arange(12,device=index.device)[None,:,None]
            score=(logmatches*aw[None,:,None,:]).sum(-1)+self.rule_logits[channel,index]
            activation=score.masked_fill(~valid,-torch.inf).softmax(-1)
            channel_beliefs=self.er(activation,beta[channel,index],dim=2)
            fusion=fw[None,:].expand(len(index),-1)
            return self.er(fusion,channel_beliefs,dim=1)
        def regularizer(self):
            return PENALTY*((self.belief_logits-self.initial_logits).square().mean()+
                self.rule_logits.square().mean()+self.attribute_logits.square().mean()+self.fusion_logits.square().mean())
    return HierarchicalBRB()

def tensors(x,refs,device):
    import torch
    idx,logm,valid=encode(x,refs)
    return (torch.as_tensor(idx,device=device,dtype=torch.long),
            torch.as_tensor(logm,device=device),torch.as_tensor(valid,device=device))

def probabilities(model,data):
    import torch
    model.eval();result=[]
    with torch.no_grad():
        for start in range(0,len(data[0]),BATCH):
            result.append(model(*(v[start:start+BATCH] for v in data)).cpu().numpy())
    p=np.concatenate(result)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=2e-6)
    return p

def fit(x,y,refs,seed,epochs,device,val=None):
    import torch
    torch.manual_seed(seed)
    if str(device).startswith('cuda'):torch.cuda.manual_seed_all(seed)
    prior=np.bincount(y,minlength=17).astype(float)+.5
    prior/=prior.sum()
    model=make_model(prior).to(device)
    initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    train=tensors(x,refs,device);target=torch.as_tensor(y,device=device,dtype=torch.long)
    validation=tensors(val[0],refs,device) if val is not None else None
    optimizer=torch.optim.Adam(model.parameters(),lr=LR)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS)
    history=[]
    for epoch in range(1,epochs+1):
        model.train();order=torch.randperm(len(y),device=device);total=0.
        for begin in range(0,len(y),BATCH):
            ix=order[begin:begin+BATCH]
            optimizer.zero_grad(set_to_none=True)
            p=model(*(v[ix] for v in train))
            loss=-p[torch.arange(len(ix),device=device),target[ix]].clamp_min(1e-12).log().mean()+model.regularizer()
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite training loss')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            optimizer.step()
            with torch.no_grad():
                model.belief_logits.clamp_(-12,12)
                model.rule_logits.clamp_(-2,2)
                model.attribute_logits.clamp_(-2,2)
                model.fusion_logits.clamp_(-2,2)
            total+=float(loss.detach())*len(ix)
        row={'epoch':epoch,'training_objective':total/len(y),'learning_rate':optimizer.param_groups[0]['lr']}
        if validation is not None:
            pred=probabilities(model,validation).argmax(1)
            row['validation_accuracy']=float(np.mean(pred==val[1]))
        history.append(row);scheduler.step()
    return model,initial,history

def select_ratio(runs,allowed):
    """Nested LDA threshold screening: outer validation rep never participates."""
    scores=[]
    for ratio in features.THRESHOLD_RATIOS:
        fold_scores=[]
        for held in allowed:
            fitting=tuple(r for r in allowed if r!=held)
            scale=features.threshold_scale(runs,fitting)
            x,meta=features.extract(runs,0,'train',scale*ratio)
            rep=meta.native_repetition.to_numpy();y=meta.gesture.to_numpy(int)
            train=np.isin(rep,fitting);validation=rep==held
            pred,_=features.predict(features.fit_lda(x[train],y[train]),x[validation])
            fold_scores.append(float(np.mean(pred==y[validation])))
        scores.append({'ratio':ratio,'mean_accuracy':float(np.mean(fold_scores)),'fold_scores':fold_scores})
    chosen=sorted(scores,key=lambda r:(-r['mean_accuracy'],abs(r['ratio']-.01)))[0]['ratio']
    return chosen,scores

def metrics(y,p):
    pred=p.argmax(1);cm=np.bincount(y*17+pred,minlength=289).reshape(17,17)
    diag=np.diag(cm);rows=cm.sum(1);cols=cm.sum(0)
    recall=np.divide(diag,rows,out=np.zeros(17,float),where=rows>0)
    f1=np.divide(2*diag,rows+cols,out=np.zeros(17,float),where=(rows+cols)>0)
    return {'accuracy':float(np.mean(pred==y)),'macro_f1':float(f1.mean()),
            'balanced_accuracy':float(recall.mean()),'windows':len(y)},cm

def export_rules(folder,model,initial,refs):
    import torch
    final={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    rows=[];fusion=[]
    for state,weights in [('initial',initial),('trained',final)]:
        model.load_state_dict(weights)
        beta,rw,aw,fw=[v.detach().cpu().numpy() for v in model.weights()]
        for c in range(12):
            fusion.append({'state':state,'channel':c+1,'fusion_weight':float(fw[c])})
            for r,bits in enumerate(BITS):
                row={'state':state,'channel':c+1,'rule':r+1,'rule_weight':float(rw[c,r])}
                for f,name in enumerate(features.FAMILIES):
                    row[f'{name}_level']=('low','medium','high')[bits[f]]
                    row[f'{name}_reference']=float(refs[f*12+c,bits[f]])
                    row[f'{name}_attribute_weight']=float(aw[c,f])
                row.update({f'belief_G{k+1}':float(beta[c,r,k]) for k in range(17)})
                rows.append(row)
    pd.DataFrame(rows).to_csv(folder/'initial_trained_rules.csv.gz',index=False)
    pd.DataFrame(fusion).to_csv(folder/'initial_trained_fusion_weights.csv',index=False)
    pd.DataFrame(refs,columns=['low','medium','high']).assign(feature=NAMES).to_csv(folder/'reference_values.csv',index=False)
    model.load_state_dict(final)
    torch.save({'state_dict':final,'initial_state_dict':initial,'references':refs.tolist()},folder/'model.pt')
    return {k:float((final[k]-initial[k]).abs().max()) for k in ['belief_logits','rule_logits','attribute_logits','fusion_logits']}

def run_subject(raw,subject,out,parent,device,max_epochs=MAX_EPOCHS,seeds=SEEDS):
    import torch
    folder=out/f'S{subject:02}';folder.mkdir(parents=True,exist_ok=True)
    runs,raw_source=features.load_subject(raw,subject)
    old=parent[str(subject)]
    assert raw_source['raw_emg_sha256']==old['raw_emg_sha256'],'Raw input differs from DB7-030'
    prepared=[];audit=[]
    for held in features.TRAIN_REPS:
        allowed=tuple(r for r in features.TRAIN_REPS if r!=held)
        ratio,screen=select_ratio(runs,allowed)
        scale=features.threshold_scale(runs,allowed)
        x,meta=features.extract(runs,subject,'train',scale*ratio)
        rep=meta.native_repetition.to_numpy();y=meta.gesture.to_numpy(int)-1
        ti=np.isin(rep,allowed);vi=rep==held
        refs=reference_values(x[ti])
        prepared.append((x[ti],y[ti],x[vi],y[vi],refs))
        audit.append({'held_repetition':held,'fitting_repetitions':allowed,'threshold_ratio':ratio,
                      'nested_threshold_scores':screen,'threshold_scale':scale.tolist(),
                      'reference_values':refs.tolist(),'train_windows':int(ti.sum()),'validation_windows':int(vi.sum())})
        print(f'S{subject:02} {device}: prepared held repetition {held}',flush=True)
    scale=features.threshold_scale(runs)
    assert np.allclose(scale,old['threshold_scale_final'],rtol=1e-10,atol=1e-15)
    xtrain,train_meta=features.extract(runs,subject,'train',scale*old['threshold_ratio'])
    ytrain=train_meta.gesture.to_numpy(int)-1
    xtest,test_meta=features.extract(runs,subject,'test',scale*old['threshold_ratio'])
    ytest=test_meta.gesture.to_numpy(int)-1
    assert len(xtest)==old['test_windows_10ms']
    refs=reference_values(xtrain)
    write(folder/'fold_audit.json',{'raw_source':raw_source,'folds':audit,
          'final_threshold_ratio':old['threshold_ratio'],'final_fitting_repetitions':features.TRAIN_REPS,
          'test_repetitions':features.TEST_REPS,'test_used_for_selection':False})
    baseline_model=features.fit_lda(xtrain,ytrain+1)
    _,lda_p=features.predict(baseline_model,xtest)
    lda_metric,lda_cm=metrics(ytest,lda_p)
    assert abs(lda_metric['accuracy']-old['lda_test_accuracy'])<1e-12,'LDA input parity failed'
    rows=[]
    for seed in seeds:
        seed_dir=folder/f'seed{seed}';seed_dir.mkdir(exist_ok=True)
        cv=[];curves=[]
        for held,(x,y,v,vy,r) in zip(features.TRAIN_REPS,prepared):
            model,_,history=fit(x,y,r,seed,max_epochs,device,val=(v,vy))
            curves.append([h['validation_accuracy'] for h in history])
            cv.extend({'held_repetition':held,**h} for h in history)
            del model
        mean_curve=np.mean(curves,axis=0)
        selected=int(np.argmax(mean_curve))+1  # tie: earliest epoch
        pd.DataFrame(cv).to_csv(seed_dir/'validation_history.csv',index=False)
        model,initial,history=fit(xtrain,ytrain,refs,seed,selected,device)
        pd.DataFrame(history).to_csv(seed_dir/'training_history.csv',index=False)
        test_data=tensors(xtest,refs,device)
        p=probabilities(model,test_data)
        result,cm=metrics(ytest,p)
        train_accuracy=float(np.mean(probabilities(model,tensors(xtrain,refs,device)).argmax(1)==ytrain))
        changes=export_rules(seed_dir,model,initial,refs)
        assert all(v>0 for v in changes.values()),'A requested parameter group did not update'
        predictions=test_meta.copy();predictions['seed']=seed
        predictions['BRB_pred']=p.argmax(1)+1;predictions['LDA_pred']=lda_p.argmax(1)+1
        predictions.to_csv(seed_dir/'predictions.csv.gz',index=False)
        np.savez_compressed(seed_dir/'probabilities.npz',brb=p,lda=lda_p.astype(np.float32))
        pd.DataFrame(cm,index=np.arange(1,18),columns=np.arange(1,18)).to_csv(seed_dir/'confusion.csv',index_label='true_gesture')
        pd.DataFrame({'gesture':np.arange(1,18),'windows':cm.sum(1),'correct':np.diag(cm),
                      'wrong':cm.sum(1)-np.diag(cm),'recall':np.diag(cm)/cm.sum(1)}).to_csv(seed_dir/'gesture_metrics.csv',index=False)
        result.update(subject=subject,seed=seed,selected_epoch=selected,validation_accuracy=float(mean_curve[selected-1]),
                      train_accuracy=train_accuracy,lda_accuracy=lda_metric['accuracy'],device=device,
                      gpu_name=torch.cuda.get_device_name(device) if str(device).startswith('cuda') else 'CPU',
                      parameter_changes=changes,success=True)
        write(seed_dir/'completion.json',result)
        rows.append({k:v for k,v in result.items() if k!='parameter_changes'})
        print(f'S{subject:02} seed{seed} {device}: epoch={selected}; BRB={result["accuracy"]:.5f}; LDA={lda_metric["accuracy"]:.5f}',flush=True)
        del model,test_data
    return rows

def self_test():
    import torch
    torch.set_num_threads(1)
    rng=np.random.default_rng(42)
    x=rng.normal(size=(85,48)).astype(np.float32);y=np.arange(85)%17
    x+=y[:,None]*.3
    refs=reference_values(x);enc=encode(x,refs)
    assert enc[0].shape==(85,12,16)
    prior=np.ones(17)/17;model=make_model(prior).double()
    with torch.no_grad():
        model.belief_logits.add_(torch.tensor(rng.normal(0,.3,(12,81,17))))
        model.rule_logits.add_(torch.tensor(rng.normal(0,.1,(12,81))))
        model.attribute_logits.add_(torch.tensor(rng.normal(0,.1,(12,4))))
    td=(torch.tensor(enc[0][:3]),torch.tensor(enc[1][:3],dtype=torch.float64),torch.tensor(enc[2][:3]))
    p=model(*td);beta,rw,aw,fw=[v.detach().numpy() for v in model.weights()]
    score=(enc[1][:3]*aw[None,:,None,:]).sum(-1)+np.log(rw[np.arange(12)[None,:,None],enc[0][:3]])
    score=np.where(enc[2][:3],score,-np.inf);a=np.exp(score-score.max(-1,keepdims=True));a/=a.sum(-1,keepdims=True)
    channels=er_numpy(a,beta[np.arange(12)[None,:,None],enc[0][:3]],2)
    expected=er_numpy(np.broadcast_to(fw,(3,12)),channels,1)
    assert np.allclose(p.detach().numpy(),expected,atol=1e-10)
    loss=-p[:,0].log().mean();loss.backward()
    for name in ['belief_logits','rule_logits','attribute_logits','fusion_logits']:
        parameter=getattr(model,name);grad=parameter.grad.flatten()
        ix=int(grad.abs().argmax());analytical=float(grad[ix]);eps=1e-5
        with torch.no_grad():parameter.view(-1)[ix]+=eps
        plus=float(-model(*td)[:,0].log().mean().detach())
        with torch.no_grad():parameter.view(-1)[ix]-=2*eps
        minus=float(-model(*td)[:,0].log().mean().detach())
        with torch.no_grad():parameter.view(-1)[ix]+=eps
        assert np.isclose(analytical,(plus-minus)/(2*eps),rtol=2e-3,atol=1e-7),name
    learned,initial,history=fit(x,y,refs,42,8,'cpu',val=(x,y))
    assert history[-1]['training_objective']<history[0]['training_objective']
    for name in ['belief_logits','rule_logits','attribute_logits','fusion_logits']:
        assert float((learned.state_dict()[name]-initial[name]).abs().max())>0,name
    print('SELF TEST PASSED: ER equivalence, all four gradient groups, learning loss, weights updated',flush=True)

def summarize(out):
    rows=[];gestures=[];devices=set()
    for subject in range(1,21):
        for seed in SEEDS:
            folder=out/f'S{subject:02}'/f'seed{seed}'
            r=json.loads((folder/'completion.json').read_text())
            assert r['success'] and all(v>0 for v in r['parameter_changes'].values())
            devices.add(r['device']);rows.append({k:v for k,v in r.items() if k!='parameter_changes'})
            gestures.append(pd.read_csv(folder/'gesture_metrics.csv').assign(subject=subject,seed=seed))
    df=pd.DataFrame(rows);df.to_csv(out/'subject_seed_metrics.csv',index=False)
    pd.concat(gestures).to_csv(out/'subject_gesture_metrics.csv',index=False)
    means=df.groupby('subject')[['accuracy','lda_accuracy']].mean()
    delta=(means.accuracy-means.lda_accuracy).to_numpy()*100
    rng=np.random.default_rng(42);boot=delta[rng.integers(0,20,(20000,20))].mean(1)
    signs=rng.choice([-1,1],(100000,20));null=np.abs((signs*delta).mean(1))
    pv=float((1+(null>=abs(delta.mean())-1e-12).sum())/(len(null)+1))
    summary={'BRB_mean_subject_accuracy_pct':float(df.accuracy.mean()*100),
        'LDA_mean_subject_accuracy_pct':float(df.lda_accuracy.mean()*100),
        'BRB_pooled_accuracy_pct':float(np.average(df.accuracy,weights=df.windows)*100),
        'LDA_pooled_accuracy_pct':float(np.average(df.lda_accuracy,weights=df.windows)*100),
        'BRB_mean_macro_f1_pct':float(df.macro_f1.mean()*100),
        'mean_gain_pp':float(delta.mean()),'gain_subject_bootstrap95_pp':np.quantile(boot,[.025,.975]).tolist(),
        'subject_signflip_p':pv,'subjects_improved':int((delta>0).sum()),'subjects_harmed':int((delta<0).sum())}
    write(out/'summary.json',summary)
    completion={'success':True,'experiment_id':'DB7-031','subjects':list(range(1,21)),'seeds':list(SEEDS),
        'final_brb_fits':60,'validation_brb_fits':240,'test_window_seed_evaluations':int(df.windows.sum()),
        'test_used_for_selection':False,'rules_per_channel':81,'channel_modules':12,'features':48,
        'references_per_feature':3,'all_parameter_groups_updated':True,'devices_used':sorted(devices)}
    assert completion['test_window_seed_evaluations']==695163 and devices=={'cuda:0','cuda:1'}
    write(out/'completion.json',completion)
    report=['# DB7-031 hierarchical BRB results','',
        'Direct EMG-only gesture classifier: 48 features, three references each,12 channel BRBs of81 rules, learned ER fusion weights.',
        'Consequent beliefs, rule weights, attribute weights and fusion weights were jointly trained. Reference values were fitted on training data, then fixed.',
        'Train1/3/4/6, test2/5;200ms;50ms training/10ms test;20 subjects;three seeds;fourfold training repetition validation.',
        'Outer-validation preprocessing uses nested training-only threshold screening; test was not used for selection.',
        'Previously inspected DB7 recordings: exploratory, not independent confirmation. Compare primarily with matched EMG-only LDA; SI also uses inertial inputs.','',
        '```json',json.dumps(summary,indent=2),'```','',
        'Initial/trained rules and weights, references, validation histories, predictions,confusions and per-gesture metrics are stored per subject/seed.']
    (out/'REPORT.md').write_text('\n'.join(report),encoding='utf-8')

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--self-test',action='store_true');parser.add_argument('--worker',action='store_true')
    parser.add_argument('--input',type=Path);parser.add_argument('--out',type=Path,default=Path('/kaggle/working/db7_031_results'))
    parser.add_argument('--parent',type=Path,default=Path('/kaggle/working/db7_031_parent.json'))
    parser.add_argument('--device',default='cuda:0');parser.add_argument('--subjects',type=int,nargs='+')
    args=parser.parse_args()
    import torch
    torch.set_num_threads(1)
    if args.self_test:self_test();return
    args.out.mkdir(parents=True,exist_ok=True)
    parent=json.loads(args.parent.read_text());raw=args.input or features.find_root()
    if args.worker:
        torch.cuda.set_device(args.device)
        write(args.out/f'worker_{args.device[-1]}_runtime.json',{'device':args.device,'gpu_name':torch.cuda.get_device_name(args.device),'pid':os.getpid(),'subjects':args.subjects,'torch_version':torch.__version__})
        for subject in args.subjects:run_subject(raw,subject,args.out,parent,args.device)
        return
    assert torch.cuda.device_count()>=2,'Two GPUs required; verify T4 x2 allocation before fitting'
    self_test()
    write(args.out/'PROTOCOL.json',{'experiment_id':'DB7-031','max_epochs':MAX_EPOCHS,'batch_size':BATCH,'optimizer':'Adam','learning_rate':LR,'regularization':PENALTY,
        'train_repetitions':features.TRAIN_REPS,'test_repetitions':features.TEST_REPS,'window_ms':200,'training_stride_ms':50,'test_stride_ms':10,
        'reference_values':'Training feature 5th/50th/95th percentiles, strictly ordered with tiny expansion for ties; fixed after fitting.',
        'architecture':'12 channel BRBs x81 rules;4 antecedents per module;17 consequent classes;trainable-weight ER fusion.',
        'learned':['consequent beliefs','rule weights','attribute weights','fusion weights'],
        'attribute_weight_constraint':'exp(logweight), normalized by channel maximum; raw logweights projected to[-2,2].',
        'rule_weight_constraint':'exp(logweight),logweight in[-2,2]; activations normalized across matching rules.',
        'epoch_selection':'Earliest epoch maximizing mean fourfold training-repetition validation accuracy, up to60; final refit on all4 training reps.',
        'fold_threshold_selection':'Nested LDA ratio screening on the three outer-fitting repetitions only; final threshold is the verified DB7-030 train-only choice.',
        'test_used_for_selection':False,'seeds':SEEDS,'subjects':list(range(1,21)),'neural_fits':0})
    workers=[]
    for device in range(2):
        subjects=list(range(device+1,21,2))
        cmd=[sys.executable,str(Path(__file__).resolve()),'--worker','--device',f'cuda:{device}',
             '--input',str(raw),'--out',str(args.out),'--parent',str(args.parent),'--subjects',*map(str,subjects)]
        workers.append(subprocess.Popen(cmd))
    codes=[p.wait() for p in workers]
    assert codes==[0,0],f'Worker failures {codes}'
    summarize(args.out)
    archive=args.out.parent/'db7_031_results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for file in sorted(args.out.rglob('*')):
            if file.is_file():z.write(file,file.relative_to(args.out))
    assert zipfile.ZipFile(archive).testzip() is None
    print('DB7-031 COMPLETE',archive,flush=True)

if __name__=='__main__':main()


## 3. Define JOPS structure and parameter learning

Variable reference memberships, constrained parameters, complete-belief ER, block Differential Evolution and structure proposals are implemented below.


In [ ]:
%%writefile /kaggle/working/db7_032_jops_engine.py
"""GPU population Differential Evolution for the DB7-032 JOPS adaptation.

This is a classification adaptation, not a reproduction of Yang et al.'s scalar
regression experiments. Structure search belongs to the caller. Each of twelve
channel modules retains four Hudgins antecedents and 17 consequent beliefs.
Every fitted quantity uses the x/y passed to train_de; test data must never be
passed there. No autograd or Adam is used by this engine.

DE jointly optimizes ordered internal references, rule weights, normalized
attribute weights, consequent beliefs, and the existing global ER fusion
weights. To make the large search practical, its initial population jitters a
training-data empirical rule-belief initialization. This initialization is an
explicit adaptation rather than a method attributed to the paper.
"""
from __future__ import annotations

import csv
import gzip
import itertools
import json
import time
from pathlib import Path

import numpy as np
import torch

FAMILIES = ("MAV", "WL", "ZC", "SSC")
N_CHANNELS, N_CLASSES = 12, 17
ACTIVE_BITS = np.asarray(list(itertools.product(range(2), repeat=4)), dtype=np.int64)
GROUPS = ("reference_logits", "rule_logits", "attribute_logits", "belief_logits", "fusion_logits")


def make_layout(counts):
    """Create a compact parameter vector. No parameters are spent on padding."""
    counts = np.asarray(counts, dtype=np.int64)
    if counts.shape == (4,):
        counts = np.tile(counts, (12, 1))
    if counts.shape != (12, 4) or not np.all((counts >= 2) & (counts <= 4)):
        raise ValueError("counts must be four values, or 12x4 values, each in [2,4]")
    rules = np.prod(counts, axis=1)
    offset, sections = 0, {}
    sizes = [int(np.where(counts > 2, counts - 1, 0).sum()), int(rules.sum()),
             48, int(rules.sum()) * 17, 12]
    for name, size in zip(GROUPS, sizes):
        sections[name] = [offset, offset + size]
        offset += size
    return {"counts": counts.tolist(), "rules_per_channel": rules.tolist(),
            "total_rules": int(rules.sum()), "dimension": offset, "sections": sections,
            "feature_order": "family-major: MAV_CH01..12, WL_CH01..12, ZC_CH01..12, SSC_CH01..12"}


def _slice(layout, name):
    return slice(*layout["sections"][name])


def parameter_bounds(layout):
    lower = np.full(layout["dimension"], -3., np.float32)
    upper = np.full(layout["dimension"], 3., np.float32)
    lower[_slice(layout, "reference_logits")] = -4.
    upper[_slice(layout, "reference_logits")] = 4.
    lower[_slice(layout, "belief_logits")] = -12.
    upper[_slice(layout, "belief_logits")] = 3.
    return lower, upper


def _validate_xy(x, y=None):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or x.shape[1] != 48 or len(x) == 0 or not np.isfinite(x).all():
        raise ValueError("x must be a finite, nonempty Nx48 array in family-major order")
    if y is None:
        return x
    y = np.asarray(y, dtype=np.int64)
    if y.shape != (len(x),) or not np.all((y >= 0) & (y < 17)):
        raise ValueError("y must contain zero-based class IDs 0..16, one per window")
    return x, y


class _Context:
    """Reusable device constants and padded population inference, without grads."""
    def __init__(self, layout, endpoints, device, dtype=torch.float32):
        self.layout, self.device, self.dtype = layout, torch.device(device), dtype
        self.counts = np.asarray(layout["counts"], int)
        self.rules = np.asarray(layout["rules_per_channel"], int)
        self.max_rules = int(max(self.rules))
        self.endpoints = torch.as_tensor(endpoints, device=device, dtype=dtype)
        self.bits = torch.as_tensor(ACTIVE_BITS, device=device)
        radix = np.stack([np.prod(self.counts[:, i + 1:], axis=1) for i in range(4)], -1)
        self.radix = torch.as_tensor(radix, device=device)
        self.n = torch.as_tensor(self.counts, device=device)
        self.channels = torch.arange(12, device=device)[None, None, :, None]
        self.reference_offsets = np.r_[0, np.cumsum(np.where(self.counts > 2, self.counts - 1, 0).sum(1))]
        self.rule_offsets = np.r_[0, np.cumsum(self.rules)]

    def decode_channel(self, vector, c):
        """Decode only one active block; avoid GPU launches for frozen modules."""
        p = len(vector)
        refs = torch.full((p, 4, 4), torch.inf, device=self.device, dtype=self.dtype)
        gap = vector[:, _slice(self.layout, "reference_logits")]
        offset = int(self.reference_offsets[c])
        for f in range(4):
            n = int(self.counts[c, f])
            lo, hi = self.endpoints[c, f]
            refs[:, f, 0] = lo
            refs[:, f, n - 1] = hi
            if n > 2:
                fractions = gap[:, offset:offset + n - 1].softmax(-1)
                fractions = 1e-4 + (1 - (n - 1) * 1e-4) * fractions
                refs[:, f, 1:n - 1] = lo + (hi - lo) * fractions.cumsum(-1)[:, :-1]
                offset += n - 1
        first, last = self.rule_offsets[c:c + 2]
        beta = vector[:, _slice(self.layout, "belief_logits")].reshape(p, -1, 17)[:, first:last].softmax(-1)
        rules = vector[:, _slice(self.layout, "rule_logits")][:, first:last]
        attrs = vector[:, _slice(self.layout, "attribute_logits")].reshape(p, 12, 4)[:, c]
        attrs = (attrs - attrs.amax(-1, keepdim=True)).exp()
        fusion = vector[:, _slice(self.layout, "fusion_logits")].softmax(-1)
        return refs, beta, rules, attrs, fusion

    def decode(self, vector):
        p = len(vector)
        refs = torch.full((p, 12, 4, 4), torch.inf, device=self.device, dtype=self.dtype)
        gap = vector[:, _slice(self.layout, "reference_logits")]
        offset = 0
        for c in range(12):
            for f in range(4):
                n = int(self.counts[c, f])
                lo, hi = self.endpoints[c, f]
                refs[:, c, f, 0] = lo
                refs[:, c, f, n - 1] = hi
                if n > 2:
                    # Positive, normalized spacings guarantee strict ordering.
                    fractions = gap[:, offset:offset + n - 1].softmax(-1)
                    fractions = 1e-4 + (1 - (n - 1) * 1e-4) * fractions
                    refs[:, c, f, 1:n - 1] = lo + (hi - lo) * fractions.cumsum(-1)[:, :-1]
                    offset += n - 1
        beta = torch.zeros((p, 12, self.max_rules, 17), device=self.device, dtype=self.dtype)
        rule_logits = torch.full((p, 12, self.max_rules), -torch.inf, device=self.device, dtype=self.dtype)
        flat_beta = vector[:, _slice(self.layout, "belief_logits")].reshape(p, -1, 17)
        flat_rule = vector[:, _slice(self.layout, "rule_logits")]
        offset = 0
        for c, r in enumerate(self.rules):
            beta[:, c, :r] = flat_beta[:, offset:offset + r].softmax(-1)
            rule_logits[:, c, :r] = flat_rule[:, offset:offset + r]
            offset += r
        attr = vector[:, _slice(self.layout, "attribute_logits")].reshape(p, 12, 4)
        attr = (attr - attr.amax(-1, keepdim=True)).exp()
        fusion = vector[:, _slice(self.layout, "fusion_logits")].softmax(-1)
        return refs, beta, rule_logits, attr, fusion

    def encode(self, x, refs):
        # x: B,48 -> P,B,12,4. Count each internal reference at/below x.
        xx = x.reshape(-1, 4, 12).transpose(1, 2)[None]
        low = (xx[..., None] >= refs[:, None, :, :, 1:]).sum(-1)
        low = torch.minimum(low, self.n[None, None] - 2)
        expanded = refs[:, None].expand(-1, len(x), -1, -1, -1)
        a = expanded.gather(-1, low[..., None]).squeeze(-1)
        b = expanded.gather(-1, (low + 1)[..., None]).squeeze(-1)
        frac = ((xx - a) / (b - a)).clamp(0, 1)
        match = torch.where(self.bits[None, None, None] == 0,
                            1 - frac[..., None, :], frac[..., None, :])
        index = ((low[..., None, :] + self.bits[None, None, None]) *
                 self.radix[None, None, :, None, :]).sum(-1)
        valid = (match > 0).all(-1)
        return index, match, valid

    @staticmethod
    def er(activation, beliefs, dim):
        singleton = ((1 + activation[..., None] * (beliefs - 1)).prod(dim) -
                     (1 - activation).prod(dim)[..., None]).clamp_min(1e-12)
        return singleton / singleton.sum(-1, keepdim=True)

    def forward(self, x, decoded, channel_output=False):
        refs, beta, rules, attrs, fusion = decoded
        index, match, valid = self.encode(x, refs)
        population = torch.arange(len(refs), device=self.device)[:, None, None, None]
        scores = (match.clamp_min(1e-30).log() * attrs[:, None, :, None, :]).sum(-1)
        scores += rules[population, self.channels, index]
        activation = scores.masked_fill(~valid, -torch.inf).softmax(-1)
        channel = self.er(activation, beta[population, self.channels, index], dim=3)
        p = self.er(fusion[:, None].expand(-1, len(x), -1), channel, dim=2)
        return (p, channel) if channel_output else p


def initialize_candidate(x, y, counts, seed=42):
    """Fit endpoints/initial references and empirical beliefs using fitting data only."""
    x, y = _validate_xy(x, y)
    layout = make_layout(counts)
    counts = np.asarray(layout["counts"])
    xx = x.reshape(-1, 4, 12).transpose(0, 2, 1)
    endpoints = np.percentile(xx, [5, 95], axis=0).transpose(1, 2, 0)
    span = np.maximum(endpoints[..., 1] - endpoints[..., 0],
                      np.maximum(np.max(np.abs(xx), axis=0) * 1e-6, 1e-12))
    equal = endpoints[..., 1] <= endpoints[..., 0]
    endpoints[..., 0] -= np.where(equal, span / 2, 0)
    endpoints[..., 1] += np.where(equal, span / 2, 0)
    vector = np.zeros(layout["dimension"], np.float32)
    gaps = []
    for c in range(12):
        for f in range(4):
            n = int(counts[c, f])
            if n > 2:
                target = np.percentile(xx[:, c, f], np.linspace(5, 95, n))
                fractions = np.maximum(np.diff(target), span[c, f] * 1e-3)
                fractions /= fractions.sum()
                raw = np.log(fractions)
                gaps.extend(raw - raw.mean())
    vector[_slice(layout, "reference_logits")] = np.clip(gaps, -4, 4)
    # Empirical initial consequents: soft counts from each window's rule activation.
    # Every class receives a half-prior pseudo-count per rule; unsupported rules
    # therefore retain a smoothed fitting-data class prior instead of test knowledge.
    prior = np.bincount(y, minlength=17).astype(float) + .5
    prior /= prior.sum()
    context = _Context(layout, endpoints, "cpu", torch.float64)
    with torch.no_grad():
        decoded = context.decode(torch.as_tensor(vector[None], dtype=torch.float64))
        refs = decoded[0]
        empirical = [np.tile(.5 * prior, (r, 1)) for r in layout["rules_per_channel"]]
        for begin in range(0, len(x), 512):
            index, match, _ = context.encode(torch.as_tensor(x[begin:begin + 512]), refs)
            ix = index[0].numpy()
            weight = match[0].prod(-1).numpy()
            for c in range(12):
                np.add.at(empirical[c], (ix[:, c].ravel(),
                          np.repeat(y[begin:begin + 512], 16)), weight[:, c].ravel())
    belief = np.concatenate([v / v.sum(-1, keepdims=True) for v in empirical])
    vector[_slice(layout, "belief_logits")] = np.log(np.maximum(belief, 1e-12)).ravel()
    lo, hi = parameter_bounds(layout)
    vector = np.clip(vector, lo, hi)
    return {"vector": vector, "initial_vector": vector.copy(), "layout": layout,
            "endpoints": endpoints, "seed": int(seed), "history": [],
            "initialization": "training empirical fuzzy-rule class counts, 0.5 prior pseudo-count per rule",
            "objective": "mean multiclass cross entropy on fitting windows", "loss": None}


def _score_vectors(vectors, x, y, context, sample_batch=256, population_chunk=4):
    result = []
    with torch.no_grad():
        for first in range(0, len(vectors), population_chunk):
            decoded = context.decode(vectors[first:first + population_chunk])
            total = torch.zeros(len(decoded[0]), device=context.device, dtype=torch.float64)
            for begin in range(0, len(x), sample_batch):
                p = context.forward(x[begin:begin + sample_batch], decoded)
                yy = y[begin:begin + sample_batch]
                ce = -p.gather(-1, yy[None, :, None].expand(len(p), -1, -1)).squeeze(-1).clamp_min(1e-12).log()
                total += ce.double().sum(-1)
            result.append(total / len(x))
    return torch.cat(result)


def train_de_global(x, y, counts, seed=42, device="cpu", population=12, generations=40,
             initial=None, sample_batch=256, population_chunk=4, mutation=.5,
             crossover=.9, jitter=.15, progress=None):
    """Optimize full fitting-set CE with bounded DE/rand/1/bin and elitist selection.

    Population and sample chunking only control memory, not which observations
    contribute to fitness. All individuals face the identical complete objective.
    The first population member is the unperturbed empirical initialization.
    Other members get independent Gaussian parameter perturbations. The seed
    controls genuine parameter initialization and subsequent DE proposals.
    """
    x, y = _validate_xy(x, y)
    if population < 4 or generations < 0 or sample_batch < 1 or population_chunk < 1:
        raise ValueError("population>=4, generations>=0 and positive chunk sizes required")
    if not 0 < mutation <= 2 or not 0 <= crossover <= 1 or jitter <= 0:
        raise ValueError("invalid DE controls")
    candidate = initialize_candidate(x, y, counts, seed) if initial is None else dict(initial)
    layout = make_layout(counts)
    if candidate["layout"]["counts"] != layout["counts"]:
        raise ValueError("A warm start must have the same structure; reinitialize changed counts")
    candidate["layout"] = layout
    context = _Context(layout, candidate["endpoints"], device)
    xx = torch.as_tensor(x, device=device, dtype=torch.float32)
    yy = torch.as_tensor(y, device=device, dtype=torch.long)
    lo, hi = [torch.as_tensor(v, device=device) for v in parameter_bounds(layout)]
    rng = np.random.default_rng(seed)
    base = np.asarray(candidate["vector"], np.float32)
    init = np.tile(base, (population, 1)) + rng.normal(0, jitter, (population, len(base))).astype(np.float32)
    init[0] = base
    vectors = torch.as_tensor(init, device=device).clamp(lo, hi)
    scores = _score_vectors(vectors, xx, yy, context, sample_batch, population_chunk)
    if not torch.isfinite(scores).all():
        raise FloatingPointError("Nonfinite initial population objective")
    initial_vector = np.asarray(base).copy()
    start = time.monotonic()
    history = [{"generation": 0, "best_training_ce": float(scores.min()),
                "mean_training_ce": float(scores.mean()), "accepted": 0, "seconds": 0.}]
    for generation in range(1, generations + 1):
        donors = np.stack([rng.choice(np.delete(np.arange(population), i), 3, replace=False)
                           for i in range(population)])
        donor = torch.as_tensor(donors, device=device)
        mutant = vectors[donor[:, 0]] + mutation * (vectors[donor[:, 1]] - vectors[donor[:, 2]])
        # Reflection repairs bounded parameters without discarding diversity.
        span = hi - lo
        repaired = lo + span - torch.abs(torch.remainder(mutant - lo, 2 * span) - span)
        mask = rng.random((population, len(base))) < crossover
        mask[np.arange(population), rng.integers(0, len(base), population)] = True
        trial = torch.where(torch.as_tensor(mask, device=device), repaired, vectors)
        trial_scores = _score_vectors(trial, xx, yy, context, sample_batch, population_chunk)
        if not torch.isfinite(trial_scores).all():
            raise FloatingPointError("Nonfinite DE trial objective")
        accepted = trial_scores <= scores
        vectors = torch.where(accepted[:, None], trial, vectors)
        scores = torch.where(accepted, trial_scores, scores)
        row = {"generation": generation, "best_training_ce": float(scores.min()),
               "mean_training_ce": float(scores.mean()), "accepted": int(accepted.sum()),
               "seconds": time.monotonic() - start}
        history.append(row)
        if progress is not None:
            progress(row)
    best = int(scores.argmin())
    vector = vectors[best].cpu().numpy()
    candidate.update(vector=vector, initial_vector=initial_vector, loss=float(scores[best]),
                     seed=int(seed), history=history, population=int(population), generations=int(generations),
                     mutation=float(mutation), crossover=float(crossover), jitter=float(jitter),
                     fitting_windows=len(x), device=str(device), optimizer="DE/rand/1/bin",
                     elapsed_seconds=time.monotonic() - start)
    candidate["parameter_changes"] = {
        name: float(np.max(np.abs(vector[_slice(layout, name)] - initial_vector[_slice(layout, name)]), initial=0))
        for name in GROUPS}
    return candidate


def _block_indices(layout):
    """One compact, jointly optimized block per channel plus a fusion block."""
    counts = np.asarray(layout["counts"])
    rules = np.asarray(layout["rules_per_channel"])
    ref_offset, rule_offset = 0, 0
    blocks = []
    for c in range(12):
        ref_size = int(np.where(counts[c] > 2, counts[c] - 1, 0).sum())
        parts = []
        for name, start, size in (("reference_logits", ref_offset, ref_size),
                                  ("rule_logits", rule_offset, rules[c]),
                                  ("attribute_logits", c * 4, 4),
                                  ("belief_logits", rule_offset * 17, rules[c] * 17)):
            first = layout["sections"][name][0] + start
            parts.extend(range(first, first + size))
        blocks.append(np.asarray(parts, dtype=np.int64))
        ref_offset += ref_size
        rule_offset += rules[c]
    blocks.append(np.arange(*layout["sections"]["fusion_logits"], dtype=np.int64))
    return blocks


def _channel_forward(context, x, decoded, c):
    """Recompute only the channel under optimization; all others are cached."""
    refs, beta, rules, attrs, _ = decoded
    if refs.ndim == 4:
        refs, beta, rules, attrs = refs[:, c], beta[:, c], rules[:, c], attrs[:, c]
    xx = x.reshape(-1, 4, 12)[:, :, c][None]
    low = (xx[..., None] >= refs[:, None, :, 1:]).sum(-1)
    low = torch.minimum(low, context.n[None, None, c] - 2)
    expanded = refs[:, None].expand(-1, len(x), -1, -1)
    a = expanded.gather(-1, low[..., None]).squeeze(-1)
    b = expanded.gather(-1, (low + 1)[..., None]).squeeze(-1)
    frac = ((xx - a) / (b - a)).clamp(0, 1)
    match = torch.where(context.bits[None, None] == 0, 1 - frac[..., None, :], frac[..., None, :])
    index = ((low[..., None, :] + context.bits[None, None]) * context.radix[None, None, c, None]).sum(-1)
    valid = (match > 0).all(-1)
    pop = torch.arange(len(refs), device=context.device)[:, None, None]
    score = (match.clamp_min(1e-30).log() * attrs[:, None, None]).sum(-1)
    score += rules[pop, index]
    activation = score.masked_fill(~valid, -torch.inf).softmax(-1)
    return context.er(activation, beta[pop, index], dim=2)


def _score_block(vectors, x, y, context, cached_channels, channel,
                 sample_batch=256, population_chunk=4):
    result = []
    with torch.no_grad():
        for first in range(0, len(vectors), population_chunk):
            v = vectors[first:first + population_chunk]
            decoded = context.decode_channel(v, channel) if channel < 12 else None
            fusion = v[:, _slice(context.layout, "fusion_logits")].softmax(-1)
            total = torch.zeros(len(v), device=context.device, dtype=torch.float64)
            for begin in range(0, len(x), sample_batch):
                cache = cached_channels[begin:begin + sample_batch][None].expand(len(v), -1, -1, -1)
                if channel < 12:
                    cache = cache.clone()
                    cache[:, :, channel] = _channel_forward(context, x[begin:begin + sample_batch], decoded, channel)
                p = context.er(fusion[:, None].expand(-1, cache.shape[1], -1), cache, dim=2)
                yy = y[begin:begin + sample_batch]
                ce = -p.gather(-1, yy[None, :, None].expand(len(p), -1, -1)).squeeze(-1).clamp_min(1e-12).log()
                total += ce.double().sum(-1)
            result.append(total / len(x))
    return torch.cat(result)


def _report_changes(candidate):
    before, after = decoded_parameters(candidate, "initial"), decoded_parameters(candidate, "trained")
    counts = np.asarray(candidate["layout"]["counts"])
    ref_change = [abs(after["references"][c, f, level] - before["references"][c, f, level])
                  for c in range(12) for f in range(4) for level in range(1, counts[c, f] - 1)]
    changes = {"reference_positions": float(max(ref_change, default=0.)),
               "rule_weights": float(np.max(np.abs(after["rule_weights"] - before["rule_weights"]))),
               "attribute_weights": float(np.max(np.abs(after["attribute_weights"] - before["attribute_weights"]))),
               "consequent_beliefs": float(np.max(np.abs(after["beliefs"] - before["beliefs"]))),
               "fusion_weights": float(np.max(np.abs(after["fusion_weights"] - before["fusion_weights"])))}
    return changes


def train_de(x, y, counts, seed=42, device="cpu", population=12, generations=4,
             cycles=2, initial=None, sample_batch=256, population_chunk=4,
             mutation=.5, crossover=.9, jitter=.15, progress=None):
    """Budgeted cooperative DE: twelve channel blocks and one fusion block.

    Each block's vector is optimized by actual DE/rand/1/bin; its fitness is the
    full classifier's CE on every fitting window. Cache the other eleven channel
    beliefs and refresh the changed channel after accepting its best candidate.
    This computational adaptation is not a claim of reproducing paper JOPS's
    global-vector optimizer. No Adam, gradients, validation or test fitness.
    The incumbent is included in every block population, preserving elitism.
    """
    x, y = _validate_xy(x, y)
    if population < 4 or generations < 0 or cycles < 1 or sample_batch < 1 or population_chunk < 1:
        raise ValueError("population>=4, generations>=0, cycles>=1, positive chunks required")
    if not 0 < mutation <= 2 or not 0 <= crossover <= 1 or jitter <= 0:
        raise ValueError("invalid DE controls")
    candidate = initialize_candidate(x, y, counts, seed) if initial is None else dict(initial)
    layout = make_layout(counts)
    if candidate["layout"]["counts"] != layout["counts"]:
        raise ValueError("Warm starts require the same structure")
    candidate["layout"] = layout
    context = _Context(layout, candidate["endpoints"], device)
    xx = torch.as_tensor(x, device=device, dtype=torch.float32)
    yy = torch.as_tensor(y, device=device, dtype=torch.long)
    bounds = [torch.as_tensor(v, device=device) for v in parameter_bounds(layout)]
    rng = np.random.default_rng(seed)
    current = torch.as_tensor(np.asarray(candidate["vector"]), device=device, dtype=torch.float32).clone()
    initial_vector = current.cpu().numpy().copy()
    blocks = _block_indices(layout)
    start = time.monotonic()
    evaluations = 0
    with torch.no_grad():
        decoded = context.decode(current[None])
        cached = torch.cat([context.forward(xx[i:i + sample_batch], decoded, channel_output=True)[1][0]
                            for i in range(0, len(xx), sample_batch)])
        current_loss = float(_score_block(current[None], xx, yy, context, cached, 12, sample_batch, 1)[0])
        initial_loss = current_loss
        initial_p = context.er(decoded[4].expand(len(xx), -1), cached, dim=1)
        initial_accuracy = float((initial_p.argmax(-1) == yy).float().mean())
        evaluations += 1
        history = [{"cycle": 0, "block": "initial", "generation": 0,
                    "best_training_ce": current_loss, "mean_training_ce": current_loss,
                    "accepted": 0, "seconds": time.monotonic() - start}]
        for cycle in range(1, cycles + 1):
            # Fixed channel order is fully reproducible and recorded explicitly.
            for channel, index_np in enumerate(blocks):
                ix = torch.as_tensor(index_np, device=device)
                base = current[ix]
                lo, hi = (v[ix] for v in bounds)
                vectors = base[None].repeat(population, 1)
                perturbation = torch.as_tensor(rng.normal(0, jitter, vectors.shape).astype(np.float32), device=device)
                vectors[1:] += perturbation[1:]
                vectors = vectors.clamp(lo, hi)

                def score(v):
                    full = current[None].repeat(len(v), 1)
                    full[:, ix] = v
                    return _score_block(full, xx, yy, context, cached, channel, sample_batch, population_chunk)

                scores = score(vectors)
                evaluations += population
                if not torch.isfinite(scores).all():
                    raise FloatingPointError("Nonfinite block initial population fitness")
                for generation in range(1, generations + 1):
                    donors = np.stack([rng.choice(np.delete(np.arange(population), i), 3, replace=False)
                                       for i in range(population)])
                    donor = torch.as_tensor(donors, device=device)
                    mutant = vectors[donor[:, 0]] + mutation * (vectors[donor[:, 1]] - vectors[donor[:, 2]])
                    span = hi - lo
                    repaired = lo + span - torch.abs(torch.remainder(mutant - lo, 2 * span) - span)
                    mask = rng.random(vectors.shape) < crossover
                    mask[np.arange(population), rng.integers(0, len(base), population)] = True
                    trial = torch.where(torch.as_tensor(mask, device=device), repaired, vectors)
                    trial_scores = score(trial)
                    evaluations += population
                    if not torch.isfinite(trial_scores).all():
                        raise FloatingPointError("Nonfinite block DE trial fitness")
                    accepted = trial_scores <= scores
                    vectors = torch.where(accepted[:, None], trial, vectors)
                    scores = torch.where(accepted, trial_scores, scores)
                    row = {"cycle": cycle, "block": f"channel_{channel+1:02}" if channel < 12 else "fusion",
                           "generation": generation, "best_training_ce": float(scores.min()),
                           "mean_training_ce": float(scores.mean()), "accepted": int(accepted.sum()),
                           "seconds": time.monotonic() - start}
                    history.append(row)
                    if progress is not None:
                        progress(row)
                winner = int(scores.argmin())
                current[ix] = vectors[winner]
                current_loss = float(scores[winner])
                if channel < 12:
                    decoded = context.decode_channel(current[None], channel)
                    for i in range(0, len(xx), sample_batch):
                        cached[i:i + sample_batch, channel] = _channel_forward(context, xx[i:i + sample_batch], decoded, channel)[0]
        # An independent full forward pass must agree with the cached objective.
        full_loss = float(_score_vectors(current[None], xx, yy, context, sample_batch, 1)[0])
        final_fusion = current[_slice(layout, "fusion_logits")].softmax(-1)
        final_p = context.er(final_fusion[None].expand(len(xx), -1), cached, dim=1)
        final_accuracy = float((final_p.argmax(-1) == yy).float().mean())
        if abs(full_loss - current_loss) > 3e-6:
            raise AssertionError(("Cached/full objective mismatch", current_loss, full_loss))
    candidate.update(vector=current.cpu().numpy(), initial_vector=initial_vector, loss=full_loss,
                     seed=int(seed), history=history, population=int(population), generations=int(generations),
                     cycles=int(cycles), mutation=float(mutation), crossover=float(crossover), jitter=float(jitter),
                     fitting_windows=len(x), device=str(device), optimizer="cooperative block DE/rand/1/bin",
                     block_order=[f"channel_{c:02}" for c in range(1, 13)] + ["fusion"],
                     searched_parameter_groups=["reference_positions", "rule_weights", "attribute_weights", "consequent_beliefs", "fusion_weights"],
                     internal_reference_degrees=int(np.maximum(np.asarray(layout["counts"]) - 2, 0).sum()),
                     fitness_evaluations=int(evaluations + 1), elapsed_seconds=time.monotonic() - start,
                     initial_training_ce=initial_loss, initial_training_accuracy=initial_accuracy,
                     final_training_ce=full_loss, final_training_accuracy=final_accuracy,
                     training_ce_reduction=initial_loss - full_loss,
                     budget_exhausted=True, convergence_claimed=False)
    candidate["raw_parameter_changes"] = {
        name: float(np.max(np.abs(candidate["vector"][_slice(layout, name)] - initial_vector[_slice(layout, name)]), initial=0))
        for name in GROUPS}
    candidate["parameter_changes"] = _report_changes(candidate)
    candidate["unchanged_parameter_groups"] = [k for k, v in candidate["parameter_changes"].items() if v == 0]
    return candidate


def predict(x, candidate, device="cpu", batch_size=512, return_channels=False):
    x = _validate_xy(x)
    context = _Context(candidate["layout"], candidate["endpoints"], device)
    outputs, channels = [], []
    with torch.no_grad():
        vector = torch.as_tensor(np.asarray(candidate["vector"])[None], device=device, dtype=torch.float32)
        decoded = context.decode(vector)
        for begin in range(0, len(x), batch_size):
            xx = torch.as_tensor(x[begin:begin + batch_size], device=device, dtype=torch.float32)
            value = context.forward(xx, decoded, channel_output=return_channels)
            if return_channels:
                value, channel = value
                channels.append(channel[0].cpu().numpy())
            outputs.append(value[0].cpu().numpy())
    p = np.concatenate(outputs)
    if not np.isfinite(p).all() or not np.allclose(p.sum(-1), 1, atol=2e-6):
        raise FloatingPointError("Invalid class probabilities")
    return (p, np.concatenate(channels)) if return_channels else p


def decoded_parameters(candidate, state="trained"):
    key = "initial_vector" if state == "initial" else "vector"
    context = _Context(candidate["layout"], candidate["endpoints"], "cpu", torch.float64)
    with torch.no_grad():
        refs, beliefs, rules, attributes, fusion = context.decode(
            torch.as_tensor(np.asarray(candidate[key])[None], dtype=torch.float64))
    return {"references": refs[0].numpy(), "beliefs": beliefs[0].numpy(),
            "rule_weights": rules[0].exp().numpy(), "attribute_weights": attributes[0].numpy(),
            "fusion_weights": fusion[0].numpy()}


def export_rules(folder, candidate):
    """Persist interpretable initial/final rules, references, weights and state."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    counts = np.asarray(candidate["layout"]["counts"])
    rules, references, weights = [], [], []
    for state in ("initial", "trained"):
        d = decoded_parameters(candidate, state)
        for c in range(12):
            weights.append({"state": state, "channel": c + 1, "fusion_weight": d["fusion_weights"][c]})
            for f, name in enumerate(FAMILIES):
                for level in range(counts[c, f]):
                    references.append({"state": state, "channel": c + 1, "attribute": name,
                                       "reference_count": counts[c, f], "level": level + 1,
                                       "reference_value": d["references"][c, f, level],
                                       "attribute_weight": d["attribute_weights"][c, f],
                                       "fitted_endpoint": level in (0, counts[c, f] - 1)})
            for r, bits in enumerate(itertools.product(*(range(n) for n in counts[c]))):
                row = {"state": state, "channel": c + 1, "rule": r + 1,
                       "rule_weight": d["rule_weights"][c, r]}
                for f, name in enumerate(FAMILIES):
                    row.update({name + "_level": bits[f] + 1,
                                name + "_reference": d["references"][c, f, bits[f]],
                                name + "_attribute_weight": d["attribute_weights"][c, f]})
                row.update({f"belief_G{k+1}": d["beliefs"][c, r, k] for k in range(17)})
                rules.append(row)
    for name, rows in [("initial_trained_rules.csv.gz", rules),
                       ("initial_trained_references.csv", references),
                       ("initial_trained_fusion_weights.csv", weights),
                       ("training_history.csv", candidate["history"])]:
        opener = gzip.open if name.endswith(".gz") else open
        with opener(folder / name, "wt", newline="", encoding="utf-8") as handle:
            if rows:
                writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
                writer.writeheader(); writer.writerows(rows)
    np.savez_compressed(folder / "model.npz", vector=candidate["vector"],
                        initial_vector=candidate["initial_vector"], endpoints=candidate["endpoints"])
    metadata = {k: v for k, v in candidate.items() if k not in ("vector", "initial_vector", "endpoints")}
    (folder / "model.json").write_text(json.dumps(metadata, indent=2, allow_nan=False), encoding="utf-8")
    return candidate.get("parameter_changes", {})


def load_candidate(folder):
    folder = Path(folder)
    result = json.loads((folder / "model.json").read_text(encoding="utf-8"))
    with np.load(folder / "model.npz", allow_pickle=False) as data:
        result.update({name: data[name] for name in ("vector", "initial_vector", "endpoints")})
    return result


save_candidate = export_rules


def _dense_numpy_prediction(x, candidate):
    """Independent Cartesian dense activation and complete-belief ER reference."""
    d = decoded_parameters(candidate)
    xx = np.asarray(x).reshape(-1, 4, 12).transpose(0, 2, 1)
    channels = []
    for c, counts in enumerate(candidate["layout"]["counts"]):
        memberships = []
        for f, n in enumerate(counts):
            refs = d["references"][c, f, :n]
            v = xx[:, c, f]
            low = np.clip(np.searchsorted(refs, v, side="right") - 1, 0, n - 2)
            frac = np.clip((v - refs[low]) / (refs[low + 1] - refs[low]), 0, 1)
            m = np.zeros((len(v), n))
            m[np.arange(len(v)), low] = 1 - frac
            m[np.arange(len(v)), low + 1] = frac
            memberships.append(m)
        bits = np.asarray(list(itertools.product(*(range(n) for n in counts))))
        a = np.ones((len(x), len(bits)))
        for f in range(4):
            a *= memberships[f][:, bits[:, f]] ** d["attribute_weights"][c, f]
        a *= d["rule_weights"][c, :len(bits)]
        a /= a.sum(-1, keepdims=True)
        beta = d["beliefs"][c, :len(bits)]
        value = np.prod(1 + a[..., None] * (beta[None] - 1), axis=1) - np.prod(1 - a, axis=1)[..., None]
        value = np.maximum(value, 1e-12)
        channels.append(value / value.sum(-1, keepdims=True))
    channel = np.stack(channels, axis=1)
    fusion = d["fusion_weights"][None, :, None]
    value = np.prod(1 + fusion * (channel - 1), axis=1) - np.prod(1 - fusion, axis=1)
    value = np.maximum(value, 1e-12)
    return value / value.sum(-1, keepdims=True)


def self_test():
    """CPU numerical parity, chunk equivalence, determinism and actual DE updates."""
    torch.set_num_threads(1)
    rng = np.random.default_rng(32032)
    x = rng.normal(size=(51, 48))
    y = np.arange(len(x)) % 17
    x += y[:, None] / 8
    maximum_error = 0.
    for counts in ([3, 3, 3, 3], [2, 3, 4, 2]):
        model = initialize_candidate(x, y, counts)
        lo, hi = parameter_bounds(model["layout"])
        model["vector"] = np.clip(model["vector"] + rng.normal(0, .2, len(lo)), lo, hi)
        context = _Context(model["layout"], model["endpoints"], "cpu", torch.float64)
        with torch.no_grad():
            decoded = context.decode(torch.as_tensor(model["vector"][None], dtype=torch.float64))
            actual = context.forward(torch.as_tensor(x), decoded)[0].numpy()
        expected = _dense_numpy_prediction(x, model)
        error = float(np.max(np.abs(actual - expected)))
        maximum_error = max(maximum_error, error)
        assert error < 1e-9, (counts, error)
        refs = decoded_parameters(model)["references"]
        for c in range(12):
            for f in range(4):
                assert np.all(np.diff(refs[c, f, :counts[f]]) > 0)
        # Compare the whole-data objective with sample and population chunking.
        vectors = torch.as_tensor(np.stack([model["vector"], model["vector"] + .02]), dtype=torch.float64)
        one = _score_vectors(vectors, torch.as_tensor(x), torch.as_tensor(y), context, 51, 2)
        chunked = _score_vectors(vectors, torch.as_tensor(x), torch.as_tensor(y), context, 7, 1)
        assert torch.allclose(one, chunked, atol=1e-12, rtol=1e-12)
    # Deliberately weak initialization tests DE learning, not an already optimal
    # empirical initializer. It is only used in this synthetic test.
    model = initialize_candidate(x, y, [3] * 4)
    model["vector"][_slice(model["layout"], "belief_logits")] = np.log(1 / 17)
    fitted = train_de(x, y, [3] * 4, seed=321, population=6, generations=2, cycles=1, initial=model,
                      sample_batch=26, population_chunk=3, jitter=.25)
    assert fitted["loss"] < np.log(17), fitted["loss"]
    assert all(v > 0 for v in fitted["parameter_changes"].values()), fitted["parameter_changes"]
    best = np.array([v["best_training_ce"] for v in fitted["history"]])
    assert np.all(np.diff(best) <= 1e-10)
    repeated = train_de(x, y, [3] * 4, seed=321, population=6, generations=2, cycles=1, initial=model,
                        sample_batch=26, population_chunk=3, jitter=.25)
    assert np.array_equal(fitted["vector"], repeated["vector"])
    result = {"passed": True, "dense_sparse_max_abs_error": maximum_error,
              "chunked_full_objective_equal": True, "deterministic": True,
              "synthetic_loss": fitted["loss"], "uniform_loss": float(np.log(17)),
              "all_five_parameter_groups_updated": fitted["parameter_changes"]}
    print(json.dumps(result, indent=2), flush=True)
    return result


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--self-test", action="store_true")
    args = parser.parse_args()
    if args.self_test:
        self_test()
    else:
        parser.print_help()


## 4. Define the experiment and diagnostics

Read the explicit configuration, fit/validation separation, dual-GPU scheduling, rule exports and completion assertions here.


In [ ]:
%%writefile /kaggle/working/db7_032_jops.py
"""DB7-032: nested repetition validation for the hierarchical JOPS adaptation.

Only fitting repetitions reach train_de. Candidate counts are selected from four
held-training-repetition folds. The test split never enters structure search.
This is an exploratory classification adaptation, not a paper reproduction.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import argparse
import hashlib
import json
import subprocess
import sys
import time
import zipfile
import numpy as np
import pandas as pd
import db7_030_hudgins as features
from db7_031_brb import select_ratio, metrics, write
import db7_032_jops_engine as engine

SEEDS = (42, 43, 44)
POPULATION, GENERATIONS, CYCLES = 12, 4, 2
CANDIDATES = 5
PARAMETER_GROUPS = ('reference_positions', 'rule_weights', 'attribute_weights',
                    'consequent_beliefs', 'fusion_weights')


def rank_key(record):
    """Prespecified validation ranking; no test quantities are accepted here."""
    return (-record['validation_accuracy'], record['validation_nll'],
            record['total_rules'], record['candidate_id'])


def pareto_archive(records):
    """SOHS-style archive: no worse accuracy and no more refs in any attribute."""
    archive = []
    for a in records:
        ac = np.asarray(a['counts'])
        dominated = False
        for b in records:
            if a['candidate_id'] == b['candidate_id']:
                continue
            bc = np.asarray(b['counts'])
            no_worse = b['validation_accuracy'] >= a['validation_accuracy']
            no_larger = np.all(bc <= ac)
            strict = b['validation_accuracy'] > a['validation_accuracy'] or np.any(bc < ac)
            if no_worse and no_larger and strict:
                dominated = True
                break
        if not dominated:
            archive.append(a)
    return sorted(archive, key=rank_key)


def propose_structure(records, rng):
    """Move counts toward a better archived structure, with bounded exploration.

    A 0.25 probability applies per attribute. Matching counts can mutate by one
    so the archive does not prevent exploration. Candidates must be distinct.
    This is a disclosed bounded SOHS-inspired proposal, not exhaustive search.
    """
    archive = pareto_archive(records)
    seen = {tuple(np.asarray(r['counts']).ravel()) for r in records}
    for attempt in range(200):
        a = archive[int(rng.integers(len(archive)))]
        b = records[int(rng.integers(len(records)))]
        better, worse = sorted([a, b], key=rank_key)
        source = np.asarray(worse['counts'], dtype=int)
        target = np.asarray(better['counts'], dtype=int)
        candidate = source.copy()
        moving = rng.random((12, 4)) < .25
        direction = np.sign(target - source)
        tied = direction == 0
        direction[tied] = rng.choice([-1, 1], int(tied.sum()))
        candidate = np.clip(candidate + moving * direction, 2, 4)
        if tuple(candidate.ravel()) not in seen:
            return candidate, {'kind': 'bounded_add_prune', 'better_candidate': better['candidate_id'],
                               'source_candidate': worse['candidate_id'], 'attempt': attempt + 1,
                               'changed_attributes': int((candidate != source).sum()), 'phi': .25}
    # Deterministic fallback avoids a duplicate even when clipping removes moves.
    base = np.asarray(archive[0]['counts'], dtype=int)
    for attribute in rng.permutation(48):
        for step in (-1, 1):
            candidate = base.copy()
            candidate.flat[attribute] = np.clip(candidate.flat[attribute] + step, 2, 4)
            if tuple(candidate.ravel()) not in seen:
                return candidate, {'kind': 'single_attribute_fallback', 'source_candidate': archive[0]['candidate_id']}
    raise RuntimeError('Unable to produce a distinct structure')


def nll(y, p):
    return float(-np.log(np.maximum(p[np.arange(len(y)), y], 1e-12)).mean())


def fit_model(x, y, counts, seed, device):
    return engine.train_de(x, y, counts, seed=seed, device=device,
                           population=POPULATION, generations=GENERATIONS, cycles=CYCLES,
                           sample_batch=2048, population_chunk=4)


def initial_model(candidate):
    return {**candidate, 'vector': candidate['initial_vector']}


def save_fit_diagnostics(folder, candidate):
    folder.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(candidate['history']).to_csv(folder / 'optimization_history.csv', index=False)
    # Initial/final rules are large, so the complete exports are limited to final
    # refits. Every validation fit retains its parameter-change and runtime audit.
    write(folder / 'fit_audit.json', {
        'layout': candidate['layout'], 'seed': candidate['seed'],
        'parameter_changes': candidate['parameter_changes'],
        'fitting_endpoints': np.asarray(candidate['endpoints']).tolist(),
        'population': POPULATION, 'generations_per_block': GENERATIONS, 'cycles': CYCLES,
        'optimizer': candidate['optimizer'], 'fitness_evaluations': candidate['fitness_evaluations'],
        'elapsed_seconds': candidate['elapsed_seconds'], 'budget_exhausted': candidate['budget_exhausted'],
        'convergence_claimed': False, 'unchanged_parameter_groups': candidate['unchanged_parameter_groups'],
        'raw_parameter_changes': candidate['raw_parameter_changes'],
        'test_used_for_fitting': False})


def run_subject(raw, subject, out, parent, device):
    import torch
    started = time.monotonic()
    folder = out / f'S{subject:02}'
    folder.mkdir(parents=True, exist_ok=True)
    runs, raw_source = features.load_subject(raw, subject)
    old = parent[str(subject)]
    assert raw_source['raw_emg_sha256'] == old['raw_emg_sha256'], 'Raw input differs from DB7-030'
    prepared, audit = [], []
    for held in features.TRAIN_REPS:
        allowed = tuple(r for r in features.TRAIN_REPS if r != held)
        ratio, screen = select_ratio(runs, allowed)
        scale = features.threshold_scale(runs, allowed)
        x, meta = features.extract(runs, subject, 'train', scale * ratio)
        rep = meta.native_repetition.to_numpy()
        y = meta.gesture.to_numpy(int) - 1
        ti, vi = np.isin(rep, allowed), rep == held
        assert not np.any(ti & vi) and np.all(ti | vi)
        prepared.append((held, x[ti], y[ti], x[vi], y[vi]))
        audit.append({'held_repetition': held, 'fitting_repetitions': allowed,
                      'threshold_ratio': ratio, 'nested_threshold_scores': screen,
                      'threshold_scale': scale.tolist(), 'train_windows': int(ti.sum()),
                      'validation_windows': int(vi.sum())})
        print(f'S{subject:02} {device}: prepared held repetition {held}', flush=True)
    scale = features.threshold_scale(runs)
    assert np.allclose(scale, old['threshold_scale_final'], rtol=1e-10, atol=1e-15)
    xtrain, train_meta = features.extract(runs, subject, 'train', scale * old['threshold_ratio'])
    ytrain = train_meta.gesture.to_numpy(int) - 1
    write(folder / 'fold_audit.json', {'raw_source': raw_source, 'folds': audit,
          'final_threshold_ratio': old['threshold_ratio'], 'final_fitting_repetitions': features.TRAIN_REPS,
          'test_repetitions': features.TEST_REPS, 'test_used_for_selection': False})

    # Freeze all structures and final parameters for this subject before reading
    # any test feature/label arrays. The raw recording naturally contains all reps.
    final_models = []
    for seed in SEEDS:
        seed_dir = folder / f'seed{seed}'
        seed_dir.mkdir(exist_ok=True)
        rng = np.random.default_rng(seed + subject * 10000)
        records = []
        for candidate_id in range(CANDIDATES):
            if candidate_id < 2:
                count = 3 if candidate_id == 0 else 2
                counts = np.full((12, 4), count, dtype=int)
                proposal = {'kind': f'prespecified_all_{count}'}
            else:
                counts, proposal = propose_structure(records, rng)
            candidate_dir = seed_dir / f'candidate{candidate_id:02}'
            candidate_dir.mkdir(exist_ok=True)
            fold_rows = []
            for held, x, y, v, vy in prepared:
                fit_start = time.monotonic()
                # Common optimization seed across structures for the same fold;
                # dimensions differ, so this is not identical initialization.
                fit_seed = seed + subject * 10000 + held * 100
                fitted = fit_model(x, y, counts, fit_seed, device)
                p = engine.predict(v, fitted, device=device)
                initial_p = engine.predict(v, initial_model(fitted), device=device)
                metric, _ = metrics(vy, p)
                save_fit_diagnostics(candidate_dir / f'held{held}', fitted)
                row = {'held_repetition': held, 'accuracy': metric['accuracy'],
                       'macro_f1': metric['macro_f1'], 'nll': nll(vy, p),
                       'initial_accuracy': float(np.mean(initial_p.argmax(1) == vy)),
                       'initial_nll': nll(vy, initial_p),
                       'fit_seconds': time.monotonic() - fit_start,
                       'fit_seed': fit_seed, 'train_windows': len(y), 'validation_windows': len(vy)}
                fold_rows.append(row)
                if candidate_id == 0 and held == features.TRAIN_REPS[0]:
                    estimate = row['fit_seconds'] * (CANDIDATES * 4 + 1) * len(SEEDS) * 10 / 3600
                    print(f'{device} timing: first full fit {row["fit_seconds"]:.1f}s; '
                          f'rough 10-subject worker estimate {estimate:.2f}h (structure sizes vary)', flush=True)
                    write(candidate_dir / 'timing_forecast.json', {'first_fit_seconds': row['fit_seconds'],
                          'rough_worker_hours': estimate, 'estimate_only': True,
                          'budget_not_changed_after_timing': True})
                    if estimate > 9:
                        raise RuntimeError('Projected worker runtime exceeds9h; stopping before the full grid. '
                                           'Inspect timing_forecast.json and shard the same protocol; do not shrink the budget silently.')
                print(f'S{subject:02} seed{seed} candidate{candidate_id} held{held}: '
                      f'val={row["accuracy"]:.5f}; seconds={row["fit_seconds"]:.1f}', flush=True)
                del fitted
            record = {'candidate_id': candidate_id, 'counts': counts.tolist(), 'proposal': proposal,
                      'total_rules': int(np.prod(counts, axis=1).sum()),
                      'validation_accuracy': float(np.mean([r['accuracy'] for r in fold_rows])),
                      'validation_nll': float(np.mean([r['nll'] for r in fold_rows])), 'folds': fold_rows}
            records.append(record)
            write(candidate_dir / 'validation.json', record)
            write(seed_dir / 'structure_search.json', {'candidates': records,
                  'pareto_archive_ids': [r['candidate_id'] for r in pareto_archive(records)],
                  'test_used_for_selection': False})
        chosen = min(records, key=rank_key)
        write(seed_dir / 'selection.json', {'selected_candidate_id': chosen['candidate_id'],
              'counts': chosen['counts'], 'total_rules': chosen['total_rules'],
              'validation_accuracy': chosen['validation_accuracy'], 'validation_nll': chosen['validation_nll'],
              'ranking': 'mean held-repetition accuracy, then NLL, then total rules, then candidate ID',
              'test_used_for_selection': False})
        final = fit_model(xtrain, ytrain, np.asarray(chosen['counts']), seed, device)
        save_fit_diagnostics(seed_dir, final)
        engine.export_rules(seed_dir, final)
        train_p = engine.predict(xtrain, final, device=device)
        initial_train_p = engine.predict(xtrain, initial_model(final), device=device)
        final_models.append((seed, final, chosen, float(np.mean(train_p.argmax(1) == ytrain)),
                             float(np.mean(initial_train_p.argmax(1) == ytrain))))

    xtest, test_meta = features.extract(runs, subject, 'test', scale * old['threshold_ratio'])
    ytest = test_meta.gesture.to_numpy(int) - 1
    assert len(xtest) == old['test_windows_10ms']
    _, lda_p = features.predict(features.fit_lda(xtrain, ytrain + 1), xtest)
    lda_metric, _ = metrics(ytest, lda_p)
    assert abs(lda_metric['accuracy'] - old['lda_test_accuracy']) < 1e-12, 'LDA input parity failed'
    for seed, final, chosen, train_accuracy, initial_train_accuracy in final_models:
        seed_dir = folder / f'seed{seed}'
        p = engine.predict(xtest, final, device=device)
        initial_test_p = engine.predict(xtest, initial_model(final), device=device)
        result, cm = metrics(ytest, p)
        changes = final['parameter_changes']
        assert all(k in changes for k in PARAMETER_GROUPS), changes
        predictions = test_meta.copy()
        predictions['seed'] = seed
        predictions['BRB_pred'] = p.argmax(1) + 1
        predictions['LDA_pred'] = lda_p.argmax(1) + 1
        predictions['BRB_correct'] = p.argmax(1) == ytest
        predictions['LDA_correct'] = lda_p.argmax(1) == ytest
        predictions.to_csv(seed_dir / 'predictions.csv.gz', index=False)
        np.savez_compressed(seed_dir / 'probabilities.npz', brb=p, initial_brb=initial_test_p,
                            lda=lda_p.astype(np.float32))
        pd.DataFrame(cm, index=np.arange(1, 18), columns=np.arange(1, 18)).to_csv(
            seed_dir / 'confusion.csv', index_label='true_gesture')
        pd.DataFrame({'gesture': np.arange(1, 18), 'windows': cm.sum(1), 'correct': np.diag(cm),
                      'wrong': cm.sum(1) - np.diag(cm), 'recall': np.diag(cm) / cm.sum(1)}).to_csv(
            seed_dir / 'gesture_metrics.csv', index=False)
        # A reference position has no free parameter when its count is two.
        searchable = {k: True for k in PARAMETER_GROUPS}
        searchable['reference_positions'] = bool(np.any(np.asarray(chosen['counts']) > 2))
        updated = all(float(changes[k]) > 0 for k in PARAMETER_GROUPS if searchable[k])
        result.update(subject=subject, seed=seed, selected_candidate=chosen['candidate_id'],
                      total_rules=chosen['total_rules'], validation_accuracy=chosen['validation_accuracy'],
                      validation_nll=chosen['validation_nll'], train_accuracy=train_accuracy,
                      initial_train_accuracy=initial_train_accuracy,
                      initial_test_accuracy=float(np.mean(initial_test_p.argmax(1) == ytest)),
                      initial_test_nll=nll(ytest, initial_test_p),
                      test_nll=nll(ytest, p), lda_accuracy=lda_metric['accuracy'],
                      device=device, gpu_name=torch.cuda.get_device_name(device),
                      parameter_changes=changes, searchable_parameter_groups=searchable,
                      all_searchable_parameters_updated=bool(updated), all_parameter_groups_searched=True,
                      validation_brb_fits=CANDIDATES * 4, structure_candidates=CANDIDATES,
                      success=True)
        write(seed_dir / 'completion.json', result)
        print(f'S{subject:02} seed{seed} {device}: JOPS={result["accuracy"]:.5f}; '
              f'LDA={lda_metric["accuracy"]:.5f}; rules={chosen["total_rules"]}', flush=True)
    write(folder / 'runtime.json', {'seconds': time.monotonic() - started, 'device': device})


def paired_subject_stats(delta):
    """Seeds averaged within subjects; windows are not independent replicates."""
    delta = np.asarray(delta, dtype=float) * 100
    rng = np.random.default_rng(42)
    boot = delta[rng.integers(0, len(delta), (20000, len(delta)))].mean(1)
    signs = rng.choice([-1, 1], (100000, len(delta)))
    null = np.abs((signs * delta).mean(1))
    p = float((1 + (null >= abs(delta.mean()) - 1e-12).sum()) / (len(null) + 1))
    return {'mean_gain_pp': float(delta.mean()), 'subject_bootstrap95_pp': np.quantile(boot, [.025, .975]).tolist(),
            'subject_signflip_p_unadjusted': p, 'subjects_improved': int((delta > 0).sum()),
            'subjects_harmed': int((delta < 0).sum())}


def summarize(out, baseline_path):
    rows, gestures, devices, changes_rows = [], [], set(), []
    for subject in range(1, 21):
        for seed in SEEDS:
            folder = out / f'S{subject:02}' / f'seed{seed}'
            r = json.loads((folder / 'completion.json').read_text())
            assert r['success'] and r['all_parameter_groups_searched']
            devices.add(r['device'])
            rows.append({k: v for k, v in r.items() if k not in ('parameter_changes', 'searchable_parameter_groups')})
            changes_rows.append({'subject': subject, 'seed': seed, **r['parameter_changes']})
            gestures.append(pd.read_csv(folder / 'gesture_metrics.csv').assign(subject=subject, seed=seed))
    df = pd.DataFrame(rows)
    baseline = pd.read_csv(baseline_path)
    assert len(baseline) == 60 and not baseline.duplicated(['subject', 'seed']).any()
    df = df.merge(baseline[['subject', 'seed', 'accuracy', 'windows']].rename(
        columns={'accuracy': 'adam_accuracy', 'windows': 'adam_windows'}), on=['subject', 'seed'],
        validate='one_to_one')
    assert len(df) == 60 and (df.windows == df.adam_windows).all()
    df.to_csv(out / 'subject_seed_metrics.csv', index=False)
    pd.DataFrame(changes_rows).to_csv(out / 'parameter_changes.csv', index=False)
    pd.concat(gestures).to_csv(out / 'subject_gesture_metrics.csv', index=False)
    means = df.groupby('subject')[['accuracy', 'lda_accuracy', 'adam_accuracy']].mean()
    means.to_csv(out / 'subject_mean_metrics.csv')
    comparisons = {name: paired_subject_stats(means.accuracy - means[column])
                   for name, column in [('JOPS_vs_Adam', 'adam_accuracy'), ('JOPS_vs_LDA', 'lda_accuracy')]}
    # Holm adjustment across these two prespecified comparator tests.
    ordered = sorted(comparisons, key=lambda k: comparisons[k]['subject_signflip_p_unadjusted'])
    running = 0.
    for index, key in enumerate(ordered):
        running = max(running, min(1., (2 - index) * comparisons[key]['subject_signflip_p_unadjusted']))
        comparisons[key]['subject_signflip_p_holm'] = running
    summary = {
        'JOPS_mean_subject_accuracy_pct': float(df.accuracy.mean() * 100),
        'JOPS_pooled_accuracy_pct': float(np.average(df.accuracy, weights=df.windows) * 100),
        'JOPS_mean_macro_f1_pct': float(df.macro_f1.mean() * 100),
        'initial_empirical_BRB_mean_subject_accuracy_pct': float(df.initial_test_accuracy.mean() * 100),
        'DE_gain_over_own_initialization_pp': float((df.accuracy - df.initial_test_accuracy).mean() * 100),
        'Adam_mean_subject_accuracy_pct': float(df.adam_accuracy.mean() * 100),
        'LDA_mean_subject_accuracy_pct': float(df.lda_accuracy.mean() * 100),
        'comparisons': comparisons, 'selected_rules_mean': float(df.total_rules.mean()),
        'non_independent_confirmation': True,
        'interpretation': 'Changes combine empirical initialization, adaptive structure, references and block DE; not an isolated optimizer ablation.'}
    write(out / 'summary.json', summary)
    completion = {'success': True, 'experiment_id': 'DB7-032', 'subjects': list(range(1, 21)),
        'seeds': list(SEEDS), 'final_brb_fits': len(df),
        'validation_brb_fits': int(df.validation_brb_fits.sum()),
        'structure_candidates': int(df.structure_candidates.sum()),
        'test_window_seed_evaluations': int(df.windows.sum()), 'test_used_for_selection': False,
        'all_parameter_groups_searched': bool(df.all_parameter_groups_searched.all()),
        'all_parameter_groups_updated': bool(df.all_searchable_parameters_updated.all()),
        'devices_used': sorted(devices), 'features': 48, 'channel_modules': 12, 'reference_count_bounds': [2, 4]}
    assert completion['test_window_seed_evaluations'] == 695163 and devices == {'cuda:0', 'cuda:1'}
    assert completion['validation_brb_fits'] == 1200 and completion['structure_candidates'] == 300
    write(out / 'completion.json', completion)
    report = ['# DB7-032 results', '',
        'Budgeted hierarchical JOPS-inspired classification adaptation on the current DB7 EMG data.',
        'Same filtered 48 Hudgins features, training1/3/4/6 and test2/5; 200ms windows, training50ms and test10ms strides.',
        'S1–S20; seeds42/43/44; 1200 validation DE fits plus60 final fits on two GPUs.',
        'Reference counts2–4, ordered internal positions, rule weights, relative attribute weights, consequent beliefs and global ER fusion weights are searched.',
        'Zero movement is reported as optimizer stagnation; two-reference attributes have no free internal position.',
        'Raw input, train-derived thresholds and test-window counts must reproduce the parent experiment.', '',
        '```json', json.dumps(summary, indent=2), '```', '',
        'Uncertainty resamples subjects after averaging seeds. Window overlap prevents treating windows as independent significance-test observations.',
        'The historical Adam comparison changes initialization, reference search, structure and optimization together; it cannot isolate one cause of improvement.',
        'Previously inspected DB7 data: exploratory evidence, not independent confirmation. Budget exhaustion is not convergence.',
        'See each subject/seed for initial/trained rules, weights, reference grids, DE history, validation selection, predictions and gesture errors.']
    (out / 'REPORT.md').write_text('\n'.join(report), encoding='utf-8')


def self_test():
    rng = np.random.default_rng(42)
    records = []
    for i, count in enumerate((3, 2)):
        counts = np.full((12, 4), count)
        records.append({'candidate_id': i, 'counts': counts.tolist(), 'total_rules': int(np.prod(counts, axis=1).sum()),
                        'validation_accuracy': .6 + i * .1, 'validation_nll': 1. - i * .1})
    assert [r['candidate_id'] for r in pareto_archive(records)] == [1]
    for i in range(2, 5):
        counts, proposal = propose_structure(records, rng)
        assert counts.shape == (12, 4) and np.all((counts >= 2) & (counts <= 4))
        assert not any(np.array_equal(counts, r['counts']) for r in records)
        records.append({'candidate_id': i, 'counts': counts.tolist(), 'total_rules': int(np.prod(counts, axis=1).sum()),
                        'validation_accuracy': .7, 'validation_nll': .9, 'proposal': proposal})
    assert min(records, key=rank_key)['candidate_id'] == 1
    assert paired_subject_stats(np.zeros(20))['subject_signflip_p_unadjusted'] == 1.
    print('ORCHESTRATOR SELF TEST PASSED: distinct bounded proposals, Pareto dominance, ranking and paired statistics', flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--self-test', action='store_true')
    parser.add_argument('--worker', action='store_true')
    parser.add_argument('--input', type=Path)
    parser.add_argument('--out', type=Path, default=Path('/kaggle/working/db7_032_results'))
    parser.add_argument('--parent', type=Path, default=Path('/kaggle/working/db7_031_parent.json'))
    parser.add_argument('--baseline', type=Path, default=Path('/kaggle/working/db7_032_adam_baseline.csv'))
    parser.add_argument('--device', default='cuda:0')
    parser.add_argument('--subjects', type=int, nargs='+')
    args = parser.parse_args()
    import torch
    torch.set_num_threads(1)
    if args.self_test:
        self_test()
        return
    args.out.mkdir(parents=True, exist_ok=True)
    parent = json.loads(args.parent.read_text())
    raw = args.input or features.find_root()
    if args.worker:
        torch.cuda.set_device(args.device)
        write(args.out / f'worker_{args.device[-1]}_runtime.json', {'device': args.device,
              'gpu_name': torch.cuda.get_device_name(args.device), 'pid': os.getpid(),
              'subjects': args.subjects, 'torch_version': torch.__version__})
        for subject in args.subjects:
            run_subject(raw, subject, args.out, parent, args.device)
        return
    assert torch.cuda.device_count() >= 2, 'Two GPUs required; verify T4 x2 before fitting'
    self_test()
    write(args.out / 'PROTOCOL.json', {'experiment_id': 'DB7-032', 'population': POPULATION,
        'generations_per_block': GENERATIONS, 'cycles': CYCLES, 'candidates_per_subject_seed': CANDIDATES,
        'sample_batch': 2048, 'population_chunk': 4,
        'runtime_guard': 'stop if first-fit extrapolated worker compute exceeds9h; shard rather than silently changing search budget',
        'optimizer': 'cooperative DE/rand/1/bin; 12 channel blocks plus fusion block',
        'objective': 'full fitting-data classification cross-entropy',
        'initialization': 'training-only empirical rule/class evidence plus jittered population',
        'structure_selection': 'mean fourfold held-training-repetition accuracy; NLL, rule count tie breakers',
        'structure_proposal': 'SOHS-inspired bounded add/prune; componentwise reference-count Pareto archive; phi0.25',
        'reference_counts': [2, 4], 'train_repetitions': features.TRAIN_REPS, 'test_repetitions': features.TEST_REPS,
        'window_ms': 200, 'training_stride_ms': 50, 'test_stride_ms': 10, 'test_used_for_selection': False,
        'learned': PARAMETER_GROUPS, 'seeds': SEEDS, 'subjects': list(range(1, 21)), 'neural_fits': 0,
        'paper_adaptation': True, 'budget_limited_not_convergence_claim': True})
    # Copy provenance into the downloadable result package.
    (args.out / 'adam_baseline.csv').write_bytes(args.baseline.read_bytes())
    provenance = args.baseline.with_suffix('.json')
    if provenance.exists():
        (args.out / 'adam_baseline_provenance.json').write_bytes(provenance.read_bytes())
    write(args.out / 'source_hashes.json', {name: hashlib.sha256((Path(__file__).parent / name).read_bytes()).hexdigest()
          for name in ('db7_030_hudgins.py', 'db7_031_brb.py', 'db7_032_jops_engine.py', 'db7_032_jops.py')})
    workers = []
    for device in range(2):
        subjects = list(range(device + 1, 21, 2))
        cmd = [sys.executable, str(Path(__file__).resolve()), '--worker', '--device', f'cuda:{device}',
               '--input', str(raw), '--out', str(args.out), '--parent', str(args.parent), '--subjects', *map(str, subjects)]
        workers.append(subprocess.Popen(cmd))
    while any(p.poll() is None for p in workers):
        if any(p.poll() not in (None, 0) for p in workers):
            for p in workers:
                if p.poll() is None:
                    p.terminate()
            break
        time.sleep(5)
    codes = [p.wait() for p in workers]
    if codes == [0, 0]:
        summarize(args.out, args.baseline)
    else:
        write(args.out / 'failure.json', {'worker_exit_codes': codes, 'success': False})
    archive = args.out.parent / 'db7_032_results.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for file in sorted(args.out.rglob('*')):
            if file.is_file():
                z.write(file, file.relative_to(args.out))
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
    assert codes == [0, 0], f'Worker failures {codes}; partial results archived'
    print('DB7-032 COMPLETE', archive, flush=True)


if __name__ == '__main__':
    main()


## 5. Pin parent provenance

These saved raw-data hashes and training-only threshold choices reproduce the parent input protocol. Parent test metrics are reproduction checks, not a selection objective.


In [ ]:
%%writefile /kaggle/working/db7_031_parent.json
{
  "1": {
    "raw_emg_sha256": "1c56de390b3f405c9e73f55560aa5ee03c2393769ae76663d9e1cedbc3353240",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00013266204041428864,
      0.0001374756684526801,
      2.5743274818523787e-05,
      1.784747109923046e-05,
      4.47269449068699e-05,
      4.1559411329217255e-05,
      6.860536814201623e-05,
      7.202202687039971e-05,
      4.7443038056371734e-05,
      0.00014688255032524467,
      1.9498520487104543e-05,
      2.1888230548938736e-05
    ],
    "test_windows_10ms": 11553,
    "lda_test_accuracy": 0.7694105427161776
  },
  "2": {
    "raw_emg_sha256": "cd05b892b99337e5c2f12969b7394cdfa09ac1f97108a451db05069089ebd7dc",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001072564409696497,
      0.00015579882892780006,
      2.5860686946543865e-05,
      1.408601110597374e-05,
      6.473016401287168e-05,
      8.070121111813933e-05,
      7.445613300660625e-05,
      8.223701297538355e-05,
      6.904384645167738e-05,
      0.0001909212296595797,
      1.370580866932869e-05,
      1.9386217900319025e-05
    ],
    "test_windows_10ms": 12695,
    "lda_test_accuracy": 0.6994092162268609
  },
  "3": {
    "raw_emg_sha256": "ed4b01bf9ce41d57fa9a8012d49720085b1c02baf6338bf35e3074c78aa5a9af",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      4.7988301957957447e-05,
      7.463988004019484e-05,
      1.5963290934450924e-05,
      1.139963114837883e-05,
      3.296114664408378e-05,
      2.861838038370479e-05,
      8.237643487518653e-05,
      4.448648905963637e-05,
      3.60179110430181e-05,
      5.011692701373249e-05,
      9.660177056503017e-06,
      7.65162258176133e-06
    ],
    "test_windows_10ms": 10087,
    "lda_test_accuracy": 0.7338158025180926
  },
  "4": {
    "raw_emg_sha256": "8a5a3fa94942dcb3cb0c1ee1b552765f8b8792bf296f6b45ad7217114ebcac0e",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.8390131016494706e-05,
      6.557466986123472e-05,
      3.7614106986438856e-05,
      2.1634888980770484e-05,
      2.2602995159104466e-05,
      4.0170711145037785e-05,
      4.2657338781282306e-05,
      3.892793756676838e-05,
      5.607752245850861e-05,
      7.70775877754204e-05,
      2.044211760221515e-05,
      3.140805347356945e-05
    ],
    "test_windows_10ms": 13860,
    "lda_test_accuracy": 0.7521645021645021
  },
  "5": {
    "raw_emg_sha256": "3b0e01332b51804a4da8770d6177e98c02d83fed03954b2a10df827663ad2b84",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      7.318822463275865e-05,
      7.545558037236333e-05,
      1.76858484337572e-05,
      1.4217462194210384e-05,
      6.991653935983777e-05,
      5.3042720537632704e-05,
      8.809279825072736e-05,
      7.663028372917324e-05,
      6.630470306845382e-05,
      7.763763278489932e-05,
      1.476779107179027e-05,
      9.451666301174555e-06
    ],
    "test_windows_10ms": 9278,
    "lda_test_accuracy": 0.7661133865057125
  },
  "6": {
    "raw_emg_sha256": "4df95dd071a2c7630bf1cde146c24009b93a576d8863fe9dccc16ac25dfe8bc8",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00010012800339609385,
      0.00011872818868141621,
      5.414242696133442e-05,
      3.222898885724135e-05,
      0.00011934531357837841,
      0.00017102088895626366,
      8.643615728942677e-05,
      8.578220149502158e-05,
      0.0001707668707240373,
      0.000146279446198605,
      2.7130547096021473e-05,
      1.381685069645755e-05
    ],
    "test_windows_10ms": 11550,
    "lda_test_accuracy": 0.7421645021645021
  },
  "7": {
    "raw_emg_sha256": "acc1ffa3d15e94345e62c23776afb21fc08a7f8888c56fde90e1a630cc3f67d7",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001515137410024181,
      0.000333660893375054,
      5.334172965376638e-05,
      3.843389276880771e-05,
      0.0002180442970711738,
      0.0001883734221337363,
      0.00015650829300284386,
      0.00013093843881506473,
      0.0001933201856445521,
      0.00031991733703762293,
      1.6768732166383415e-05,
      8.15492239780724e-05
    ],
    "test_windows_10ms": 9383,
    "lda_test_accuracy": 0.7490141745710327
  },
  "8": {
    "raw_emg_sha256": "ce9071373566c7db8bdc0be56e8c645930ac9b47be1b28e80d88e9d55d4028aa",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001147155198850669,
      0.00027329017757438123,
      3.581086275517009e-05,
      1.6894049622351304e-05,
      6.151852721814066e-05,
      9.829152259044349e-05,
      9.444140596315265e-05,
      6.824731826782227e-05,
      5.348913691705093e-05,
      0.00013976218178868294,
      9.565607797412667e-06,
      1.2080162377969828e-05
    ],
    "test_windows_10ms": 11886,
    "lda_test_accuracy": 0.767794043412418
  },
  "9": {
    "raw_emg_sha256": "bee0940a4ffe3f252fbc07ec3e0e0afd3b67d796e5ae333bc718e2b9f3de07ce",
    "threshold_ratio": 0.005,
    "threshold_scale_final": [
      0.00010828717495314777,
      0.0001047237019520253,
      2.7731381123885512e-05,
      1.8029701095656492e-05,
      0.00011607875057961792,
      9.820485138334334e-05,
      4.905431705992669e-05,
      5.873566624359228e-05,
      7.305116741918027e-05,
      0.00013037046301178634,
      1.738184437272139e-05,
      3.154447040287778e-05
    ],
    "test_windows_10ms": 10305,
    "lda_test_accuracy": 0.7412906356137797
  },
  "10": {
    "raw_emg_sha256": "1e46abf381b0056ac17923dd7612be87c4681eb9c32fc4d7c2f1459fcc7d39c9",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001176421792479232,
      0.00012255931505933404,
      0.00010596213542157784,
      3.420419670874253e-05,
      9.902081364998594e-05,
      9.533303091302514e-05,
      7.600613025715575e-05,
      7.306517363758758e-05,
      0.00012148161476943642,
      0.00010842792835319415,
      1.7887134163174778e-05,
      2.2611628082813695e-05
    ],
    "test_windows_10ms": 11750,
    "lda_test_accuracy": 0.6695319148936171
  },
  "11": {
    "raw_emg_sha256": "f11d5dfdf6092433e71ab7e88b93ea0810a9e1effd5a6e8ff51b1b2c50d33722",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.514318399946205e-05,
      8.086879824986681e-05,
      3.407187614357099e-05,
      1.5211981008178554e-05,
      3.629827551776543e-05,
      2.9886939955758862e-05,
      2.5073355573113076e-05,
      4.232222636346705e-05,
      5.541784412343986e-05,
      5.866068386239931e-05,
      1.6799922377686016e-05,
      1.605762190592941e-05
    ],
    "test_windows_10ms": 14244,
    "lda_test_accuracy": 0.689623701207526
  },
  "12": {
    "raw_emg_sha256": "afa6eb0517c388c28f458886332cb96be4469e3376731aaced584f1b9594b56f",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00010118428326677531,
      0.00021223636576905847,
      3.55233351001516e-05,
      1.3748856872553006e-05,
      8.230475941672921e-05,
      0.0001146679132943973,
      0.00011095914669567719,
      6.208808190422133e-05,
      9.482594032306224e-05,
      9.964436321752146e-05,
      7.21825344953686e-05,
      9.099677299673203e-06
    ],
    "test_windows_10ms": 12272,
    "lda_test_accuracy": 0.6676988265971316
  },
  "13": {
    "raw_emg_sha256": "0e55d064adfe43be83738eb22ab99bcfa19fda45bbd0048c1bac382575e89282",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.280390178086236e-05,
      9.842139843385667e-05,
      6.807075988035649e-05,
      1.2225775208207779e-05,
      2.12231716432143e-05,
      9.242261876352131e-05,
      5.533037619898096e-05,
      5.869295273441821e-05,
      9.206052345689386e-05,
      0.00011729584366548806,
      3.182988439220935e-05,
      8.137374607031234e-06
    ],
    "test_windows_10ms": 12707,
    "lda_test_accuracy": 0.6941056110805068
  },
  "14": {
    "raw_emg_sha256": "602afca1f5de11d0ad728f4414394f38d0c1d90659d4507b692c97a9ccc54311",
    "threshold_ratio": 0.01,
    "threshold_scale_final": [
      0.00013264398148749024,
      0.00021044100867584348,
      4.297760460758582e-05,
      1.7475626009400003e-05,
      0.00011667045328067616,
      7.852225826354697e-05,
      0.00011536300007719547,
      0.00016484233492519706,
      0.00011176728730788454,
      8.786814578343183e-05,
      2.2566648112842813e-05,
      1.3923663573223166e-05
    ],
    "test_windows_10ms": 11584,
    "lda_test_accuracy": 0.7045925414364641
  },
  "15": {
    "raw_emg_sha256": "bf717ea71162a5cf01445393c3a20d66bf3af2b21cf8d926161d9a34a8ee3c28",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      6.782060518162325e-05,
      0.00016464098007418215,
      2.5319724954897538e-05,
      1.2249682185938582e-05,
      5.61313281650655e-05,
      3.773471325985156e-05,
      3.8305443013086915e-05,
      3.9486305468017235e-05,
      6.851983198430389e-05,
      5.4393927712226287e-05,
      1.3233364370535128e-05,
      1.5889450878603384e-05
    ],
    "test_windows_10ms": 11455,
    "lda_test_accuracy": 0.7030117852466172
  },
  "16": {
    "raw_emg_sha256": "c02de75b4bc2ede5715b71d704f4969ddfef14de442e407b6559991c898a0868",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00019861815962940454,
      0.00016362268070224673,
      7.93134022387676e-05,
      2.511195998522453e-05,
      3.445372567512095e-05,
      6.207184924278408e-05,
      7.871009438531473e-05,
      0.00015840321430005133,
      8.483541023451835e-05,
      0.00017966260202229023,
      1.3469390978571028e-05,
      1.391061505273683e-05
    ],
    "test_windows_10ms": 11385,
    "lda_test_accuracy": 0.7006587615283267
  },
  "17": {
    "raw_emg_sha256": "3c819fbca8446569bd2c31fb49959d7796a49d0762304f24f8134af73eb73317",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.337742550182156e-05,
      0.0001356772263534367,
      5.898009112570435e-05,
      1.3639853023050819e-05,
      3.406200994504616e-05,
      5.478320235852152e-05,
      5.0377362640574574e-05,
      2.9153767172829248e-05,
      5.622144090011716e-05,
      0.00012032186350552365,
      2.2048187020118348e-05,
      8.751561836106703e-06
    ],
    "test_windows_10ms": 11039,
    "lda_test_accuracy": 0.7352115227828607
  },
  "18": {
    "raw_emg_sha256": "f6a6081a3fc28fc7ac0b5e40b82e1206cf881d3c1ee4fb8192772cce773bdf93",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      6.631188443861902e-05,
      8.188419451471418e-05,
      2.2324860765365884e-05,
      9.289688023272902e-06,
      1.9854758647852577e-05,
      4.6616514737252146e-05,
      2.153000059479382e-05,
      5.2621340728364885e-05,
      4.335492121754214e-05,
      4.028893090435304e-05,
      9.028473868966103e-06,
      1.9218265151721425e-05
    ],
    "test_windows_10ms": 13052,
    "lda_test_accuracy": 0.7860098069261416
  },
  "19": {
    "raw_emg_sha256": "3505bf92ff6501279405b4b6988a972f54bfe7493e3f3f88c19eef571ab7cd8a",
    "threshold_ratio": 0.005,
    "threshold_scale_final": [
      9.851076902123168e-05,
      0.0001474832242820412,
      3.4470394894015044e-05,
      1.9086524844169617e-05,
      5.6170196330640465e-05,
      8.336344035342336e-05,
      9.021452569868416e-05,
      5.269997564028017e-05,
      4.7443387302337214e-05,
      8.203284232877195e-05,
      2.193837644881569e-05,
      1.8382097550784238e-05
    ],
    "test_windows_10ms": 10975,
    "lda_test_accuracy": 0.6433712984054669
  },
  "20": {
    "raw_emg_sha256": "58bb627d65f3cd87a3c434f781126589eea47d19727cc20e54c5b701ad6ddf72",
    "threshold_ratio": 0.01,
    "threshold_scale_final": [
      0.00014526289305649698,
      0.00019448946113698184,
      0.0004191744374111295,
      4.356720091891475e-05,
      2.549423516029492e-05,
      0.00016596984642092139,
      0.00010782211029436439,
      6.814220250817016e-05,
      0.0001349761732853949,
      0.0001800787722459063,
      3.6949502828065306e-05,
      2.0891255189781077e-05
    ],
    "test_windows_10ms": 10661,
    "lda_test_accuracy": 0.6287402682675172
  }
}

## 6. Retain the historical DB7-031 result

These 60 subject/seed metrics are used only for the final report. They are never used to select a structure or parameter setting.


In [ ]:
%%writefile /kaggle/working/db7_032_adam_baseline.csv
subject,seed,accuracy,macro_f1,windows
1,42,0.4678438500822297,0.3981960329452908,11553
1,43,0.4695750021639401,0.4013885033186616,11553
1,44,0.4707868086211373,0.3970673477312157,11553
2,42,0.4569515557306026,0.3783678233649787,12695
2,43,0.4764868058290665,0.4380385701210069,12695
2,44,0.4741236707365104,0.4352174604136524,12695
3,42,0.5472390205214632,0.5426348752571534,10087
3,43,0.5494200456032517,0.5457312332858978,10087
3,44,0.5522950332110638,0.5492336677525358,10087
4,42,0.454978354978355,0.3442973471793008,13860
4,43,0.4547619047619047,0.3440409961807638,13860
4,44,0.4537518037518037,0.3455987952708089,13860
5,42,0.4535460228497521,0.3775465774726671,9278
5,43,0.4624919163612848,0.3784760431905274,9278
5,44,0.4527915499029963,0.3769336073847391,9278
6,42,0.5052813852813853,0.4857397977944344,11550
6,43,0.5010389610389611,0.4820967945428566,11550
6,44,0.5051082251082251,0.4845361317135581,11550
7,42,0.4714909943514867,0.4286036583461606,9383
7,43,0.4779921133965683,0.4412188495850302,9383
7,44,0.4754342960673559,0.4334264080529872,9383
8,42,0.414268887767121,0.3520811846716623,11886
8,43,0.4098098603398957,0.344925161602796,11886
8,44,0.425206124852768,0.3661355254022951,11886
9,42,0.4934497816593886,0.492805455635837,10305
9,43,0.4952935468219311,0.4944869338485541,10305
9,44,0.4977195536147501,0.4958598896581942,10305
10,42,0.4661276595744681,0.4428255233852134,11750
10,43,0.4654468085106383,0.4409877002261745,11750
10,44,0.4714893617021277,0.448162587214468,11750
11,42,0.5050547598989048,0.4786716459348873,14244
11,43,0.4877843302443134,0.4625970582711182,14244
11,44,0.4898904802021904,0.4648459959823222,14244
12,42,0.3702737940026075,0.3203874253263745,12272
12,43,0.393089960886571,0.3423219328986582,12272
12,44,0.4269067796610169,0.3736781833335431,12272
13,42,0.3755410403714488,0.3166763287774788,12707
13,43,0.3171480286456284,0.2630304316986666,12707
13,44,0.314787125206579,0.2566701484358922,12707
14,42,0.480835635359116,0.4553130883836002,11584
14,43,0.4820441988950276,0.4571776045268136,11584
14,44,0.4804040055248619,0.4524792539132212,11584
15,42,0.4746398952422523,0.3741869902986419,11455
15,43,0.469402007856831,0.3725135828898236,11455
15,44,0.4585770405936272,0.3421732655734027,11455
16,42,0.4929292929292929,0.4535022377622769,11385
16,43,0.4968818620992534,0.4589360664107759,11385
16,44,0.4908212560386473,0.4534047162662932,11385
17,42,0.5221487453573693,0.4904770847205857,11039
17,43,0.5572968565993297,0.5263126573777148,11039
17,44,0.5123652504755866,0.4770495333438099,11039
18,42,0.4557922157523751,0.3834123067723937,13052
18,43,0.4588568801716212,0.4016394275161196,13052
18,44,0.4543365001532332,0.3848982904192684,13052
19,42,0.4431890660592255,0.4174765743781654,10975
19,43,0.44,0.4132424198349007,10975
19,44,0.443006833712984,0.4155467407757235,10975
20,42,0.4760341431385423,0.4541950592381727,10661
20,43,0.4702185536066035,0.4503868396379844,10661
20,44,0.4681549573210768,0.4461120185512277,10661


## 7. Record historical-result provenance

The provenance record identifies the saved DB7-031 metrics and their integrity hash.


In [ ]:
%%writefile /kaggle/working/db7_032_adam_baseline.json
{
  "source_experiment": "DB7-031",
  "run_id": 36421407246,
  "notebook_sha256": "a17e8d82ba9bbc4703f28c12c9202da8d03602ba8c7021908a8a4e3fc03a6270",
  "metrics_source_sha256": "de22b3866e59d04d7215ac89c4a9e68ae755ad7cdc360e0dc8897951f181d56e",
  "role": "Historical matched-data Adam baseline for reporting only; never structure or parameter selection"
}

## 8. Run checks, then fit using both GPUs

The engine and orchestration self-tests run before any full model training.
A missing GPU, data mismatch,
failed fit or incomplete output aborts execution. Parameter-change evidence is
recorded across the run; every parameter group must be searched, but DE is allowed
to record stagnation rather than fabricating an update. Initial and final exports
make this inspectable. A two-reference feature has no internal reference position.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
subprocess.run([sys.executable, '/kaggle/working/db7_032_jops_engine.py', '--self-test'], check=True)
subprocess.run([sys.executable, '/kaggle/working/db7_032_jops.py', '--self-test'], check=True)
subprocess.run([sys.executable, '/kaggle/working/db7_032_jops.py'], check=True)
out = Path('/kaggle/working/db7_032_results')
done = json.loads((out / 'completion.json').read_text())
assert done['success']
assert done['subjects'] == list(range(1, 21)) and done['seeds'] == [42, 43, 44]
assert done['final_brb_fits'] == 60 and done['validation_brb_fits'] == 1200
assert done['structure_candidates'] == 300
assert done['devices_used'] == ['cuda:0', 'cuda:1']
assert done['test_window_seed_evaluations'] == 695163
assert done['all_parameter_groups_searched'] and not done['test_used_for_selection']
assert isinstance(done['all_parameter_groups_updated'], bool)
print(json.dumps(done, indent=2))


## 9. Inspect measured results

The table reports each final subject/seed model. The plot averages seeds within
each subject. Gesture recall counts actual predictions; the test grid is dense
and overlapping. Read the accompanying report for paired uncertainty and limits.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

display(Markdown((out / 'REPORT.md').read_text()))
metrics = pd.read_csv(out / 'subject_seed_metrics.csv')
display(metrics)
by_subject = metrics.groupby('subject')[['accuracy', 'lda_accuracy']].mean() * 100
by_subject.columns = ['JOPS BRB', 'Matched Hudgins LDA']
ax = by_subject.plot.bar(figsize=(14, 5), ylim=(0, 100), ylabel='Test accuracy (%)',
                         title='DB7-032: JOPS BRB and matched LDA')
plt.tight_layout()
plt.savefig(out / 'subject_accuracy.png', dpi=160)
plt.show()

gestures = pd.read_csv(out / 'subject_gesture_metrics.csv')
grid = gestures.groupby(['subject', 'gesture']).recall.mean().unstack() * 100
fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(grid, vmin=0, vmax=100, aspect='auto', cmap='viridis')
ax.set(xticks=range(17), xticklabels=range(1, 18), yticks=range(20),
       yticklabels=range(1, 21), xlabel='Gesture', ylabel='Subject',
       title='JOPS BRB mean test recall by subject and gesture')
fig.colorbar(im, ax=ax, label='Recall (%)')
fig.tight_layout()
fig.savefig(out / 'gesture_recall.png', dpi=160)
plt.show()

# Include presentation plots in the result ZIP downloaded by GitHub Actions.
import zipfile
with zipfile.ZipFile('/kaggle/working/db7_032_results.zip', 'w',
                     zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file in sorted(out.rglob('*')):
        if file.is_file():
            archive.write(file, file.relative_to(out))
print('Download db7_032_results.zip for all rules, weights, histories and predictions.')
